# Post-training XLS-R deepfake detector (CAC) — v4: ElevenLabs-targeted, self-correcting fine-tune

This notebook **continues training V3** (`export_posttrain_v3/detector_state.pt`). It doesn't start over. It reuses the V2 cache (MLAAD fakes + M-AILABS real) and the V3 LibriSpeech cache already on Drive.

**Why a fourth round.** The October 23 comparison (`model teseting/Compare V2 and V3/`) found:

| Finding | V2 | V3 |
|---|---:|---:|
| ElevenLabs windows detected (each model's EER threshold) | 99.6% | 79.1% |
| Human windows falsely flagged | 57.1% | 2.0% |
| ElevenLabs detected under 10 dB white noise | 96.4% | 42.9% |
| ElevenLabs calls the app labels "AI" (call replay) | 27/30 | 0/30 |
| Non-speech probes flagged | 41/45 | 0/45 |

V3 fixed false alarms but lost ElevenLabs, mostly under noise. **No model so far has seen any ElevenLabs audio in training.** Without white noise, V3 already separates the classes. Noise and lack of ElevenLabs data are the two things to fix.

## What V4 changes

1. **ElevenLabs (and other commercial voice-clone) audio in training,** split by voice so test voices are never trained on. Two data modes are chosen automatically (section 7):
   - **external** (recommended): the public *Fake Audio Dataset (ElevenLabs & Respeecher)*. Your 3 benchmark voices (Shrey, Keshav, Azim) stay a **true held-out test**.
   - **own-voices fallback**: only your clips. One voice (default Shrey, the hardest) is held out for test, and the others train. The repo benchmark is then no longer independent for the trained voices, and the notebook says so.
2. **Noise-consistency loss.** Every batch also scores a noisier copy of part of the batch (0–20 dB SNR). The noisy logit is pulled toward the clean one. Labels aren't involved, so it can't become a shortcut. This targets the white-noise collapse.
3. **One-sided distillation from V3 ("never worse than V3").** A frozen V3 copy scores each batch. The loss penalizes V4 only when it scores real speech as *more* AI than V3 did, or a known MLAAD fake as *less* AI than V3 did. Improvements are free; regressions cost.
4. **Anchor to V3 weights.** Each step pulls weights slightly back toward V3 to limit forgetting.
5. **Label-blind source augmentation.** Wideband MP3 and band-limit round trips on *both* classes, so "sounds like an ElevenLabs MP3" can't become a shortcut.
6. **A self-correcting training loop (the Guardian),** described next.
7. **Selection on the low-false-alarm end:** ElevenLabs detection at a 2% real-speech false-flag threshold, not EER.
8. **Final test plus release gate,** a README-style **V2 vs V3 vs V4** benchmark table, and a replay through the app's call-verdict logic. The export also includes the recommended call-engine thresholds.

## The Guardian (self-correction)

About 3 times per epoch, the Guardian scores the model on **validation** data:

- each robustness condition's EER
- pooled EER
- detection of other TTS systems at 2% false flags
- LibriSpeech (unseen speakers) false flags
- the non-speech probes
- the **ElevenLabs objective**

Every metric must stay within a small tolerance of the **best accepted state**, which starts as V3:

| Result of a check | What the Guardian does |
|---|---|
| All guards pass and the ElevenLabs objective improves | Saves a new best (ratchet) |
| Any guard regresses, or the ElevenLabs objective drops | **Rolls back** to the best weights, halves the learning rate, resets the optimizer, and **retunes the cause** (below) |
| Stalled (no gain for a while) | Raises the ElevenLabs sampling share |
| Correction budget used up | Stops and keeps the best accepted state |

How the cause is retuned:

| Regressed metric | Retuning |
|---|---|
| Noise | More noise augmentation, stronger consistency loss |
| Silence gaps | More gap augmentation |
| Music | More background-overlay augmentation |
| Clean or other-TTS | Stronger distillation and anchor, less ElevenLabs pressure |
| Non-speech | More non-speech examples labeled not-AI |

Every decision is written to `self_correction_log.json`.

**Honesty rules built in:** the Guardian only ever sees *validation* data. Test data and your benchmark recordings are scored once, after training, and never steer anything. If no checkpoint beats V3 on every guard, the kept state is V3 itself, and the export says so. If the final test **release gate** fails, the export goes to a folder marked `CANDIDATE_DO_NOT_DEPLOY`.

## Data to put on Drive (`MyDrive/CAC_deepfake_detector/`)

| Folder | Needed? | What |
|---|---|---|
| `cache_en/` | yes (exists if V2 ran) | V2 cache |
| `cache_extra_real_v3/` | yes (exists if V3 ran; rebuilt if missing) | LibriSpeech cache |
| `export_posttrain_v3/detector_state.pt` | yes | V3 checkpoint (SHA-256 `c634c95b…`) |
| `ckpt_en/best.pt` | optional | V2, used only in the comparison table |
| `external/elevenlabs_respeecher_mendeley/` | **strongly recommended** | Download **Fake Audio Dataset (ElevenLabs & Respeecher)** from <https://data.mendeley.com/datasets/79g59sp69z> (Download All → unzip here, keep the Excel metadata). CC BY 4.0: credit Ballesteros & Beltran, Universidad Militar Nueva Granada. The page doesn't state the language; the notebook reports results by source so you can see whether it helps English ElevenLabs. |
| `benchmark_audio/` | recommended | The 5 benchmark files from the repo (`model teseting/Compare V2 and V3/inputs/audio/`: 3 ElevenLabs MP3s + 2 LibriSpeech WAVs), for the README-style table |
| `own_data/elevenlabs/<voice>/` | optional | Any *extra* ElevenLabs clips you generate with your voices (different scripts help even with the same 3 voices) |
| `own_data/real/<speaker>/` | optional, valuable | Consented real recordings, e.g. the source audio your clones were made from. These are used in training, split by speaker. |

## How to run (Colab Pro)

1. `Runtime → Change runtime type → L4` (or A100).
2. `Runtime → Run all`, approve Drive.
3. **Rough time on an L4: about 2–3 h** (data prep 10–20 min, up to 4 epochs with Guardian checks, final evaluation about 20 min). This is an estimate: the twin views and the V3 teacher make each step about 1.6× V3's cost. It hasn't been timed on a GPU.
4. If the runtime disconnects, **Run all** again: caches, the best state, the Guardian state and the training position resume from Drive.

**Licenses:** MLAAD CC BY-NC 4.0, LibriSpeech CC BY 4.0, ESC-50 CC BY-NC, the Mendeley set CC BY 4.0. Fine for a CAC entry.

## 1. Configuration
Defaults are starting points, not tuned values. The Guardian adjusts the knobs marked *(knob)* during training, within `KNOB_LIMITS`.

In [ ]:
CFG = dict(
    # ---- start from V3 ----
    BASE_MODEL       = "facebook/wav2vec2-xls-r-300m",
    KEEP_LAYERS      = 12,
    ARCH_CONFIG_JSON = None,          # None = Wav2Vec2Config.from_pretrained(BASE_MODEL)
    INIT_CHECKPOINT  = "export_posttrain_v3/detector_state.pt",   # relative paths are inside DRIVE_DIR
    EXPECTED_INIT_SHA256 = "c634c95b2bb0fb5f5505d4d0364ad96980aa00de83d93bcc04a34ec4585e96a8",   # V3 detector_state.pt
    V3_LOGIT_THRESHOLDS = (2.03125, 6.000000000000001),          # V3 config: logit_threshold_eer, logit_threshold_1pct_fpr
    V2_CHECKPOINT    = "ckpt_en/best.pt",                          # optional, comparison only
    V2_LOGIT_THRESHOLDS = (11.560348981416222, 15.942385033669446),  # as used by the Compare V2/V3 run
    # ---- audio (must match the live agent) ----
    SR = 16000, TRAIN_SEC = 4.0, MAX_STORE_SEC = 8.0, MIN_SEC = 1.0, SEGMENT_SEC = 8.0,
    # ---- optimisation ----
    MAX_EPOCHS = 4, CHECKS_PER_EPOCH = 3,
    LR_BACKBONE = 3e-6, LR_HEAD = 1e-4, WEIGHT_DECAY = 1e-4, WARMUP_FRAC = 0.05, GRAD_CLIP = 1.0,
    # ---- losses (knobs) ----
    CONSIST_WEIGHT = 0.5,             # (knob) noisy-twin consistency
    CONSIST_FRAC   = 0.5,             # share of each batch that also gets a noisy twin
    CONSIST_SNR_DB = (0.0, 20.0),
    KD_WEIGHT      = 1.0,             # (knob) one-sided "never worse than V3" distillation
    ANCHOR_RATE    = 2e-4,            # (knob) per-step pull of weights back toward V3 (x LR schedule factor)
    # ---- sampling (knobs) ----
    ELEVEN_FAKE_SHARE      = 0.20,    # (knob) share of fake draws that are ElevenLabs
    OTHER_COMMERCIAL_SHARE = 0.05,    # Respeecher etc.
    OWN_REAL_SHARE         = 0.10,    # share of real draws from own_data/real (if any)
    # ---- ElevenLabs / commercial data ----
    ELEVEN_OWN_SUBDIR      = "own_data/elevenlabs",
    ELEVEN_EXTERNAL_SUBDIR = "external/elevenlabs_respeecher_mendeley",
    USE_RESPEECHER         = True,
    MIN_EXTERNAL_ELEVEN    = 50,      # fewer ElevenLabs clips than this in the external folder -> own-voices fallback
    BENCHMARK_SUBDIR       = "benchmark_audio",
    BENCHMARK_VOICES       = ["shrey", "keshav", "azim"],   # never trained on in "external" mode
    FALLBACK_TEST_VOICES   = ["shrey"],                     # held out in the own-voices fallback
    ELEVEN_VAL_FRAC = 0.2, ELEVEN_TEST_FRAC = 0.2,
    OWN_REAL_SUBDIR = "own_data/real", OWN_EVAL_SUBDIR = "own_data/eval_only",
    # ---- augmentation (label-blind; knobs marked) ----
    PHONE_AUG_P = 0.7, OPUS_P = 0.12, MP3_P = 0.10,
    NOISE_P = 0.65,                   # (knob)
    NOISE_SNR_DB = (0.0, 30.0), RECORDED_NOISE_FRAC = 0.35, WHITE_NOISE_FRAC = 0.4,
    OVERLAY_P = 0.25,                 # (knob)
    OVERLAY_SNR_DB = (5.0, 25.0),
    GAP_P = 0.35,                     # (knob)
    GAP_SEC = (0.1, 2.5), GAP_DIGITAL_FRAC = 0.5,
    NOISE_FLOOR_P = 0.35, NOISE_FLOOR_DBFS = (-75.0, -50.0), NOISE_GATE_P = 0.15, CLIP_P = 0.08,
    NONSPEECH_AS_REAL_P = 0.03,       # (knob)
    WB_MP3_P = 0.12, BANDLIMIT_P = 0.15,   # new: wideband MP3 / band-limit round trips on both classes
    # ---- Guardian ----
    TARGET_FPR = 0.02,                # objective & guards measured at this real-speech false-flag rate
    TOL_POOLED_EER = 0.005, TOL_COND_EER = 0.01, TOL_TPR = 0.01, TOL_LS_FPR = 0.01, TOL_NONSPEECH = 0.0125, TOL_OBJECTIVE = 0.03,
    MIN_OBJECTIVE_GAIN = 0.002, MAX_CORRECTIONS = 4, MAX_BOOSTS = 3, STALL_CHECKS = 2, LR_BACKOFF = 0.5,
    VAL_CHECK_MAX_CLIPS = 2000,       # stratified subsample of validation used at each check
    SELECT_CONDITIONS = ["clean", "phone", "noise20", "noise10", "gap2s", "music10"],
    KNOB_LIMITS = dict(NOISE_P=(0.4, 0.9), GAP_P=(0.2, 0.6), OVERLAY_P=(0.15, 0.5), NONSPEECH_AS_REAL_P=(0.0, 0.1),
                       CONSIST_WEIGHT=(0.1, 2.0), KD_WEIGHT=(0.25, 4.0), ANCHOR_RATE=(0.0, 1e-3),
                       ELEVEN_FAKE_SHARE=(0.05, 0.40), LR_SCALE=(0.05, 1.0)),
    # ---- LibriSpeech (only if the V3 cache is missing) / ESC-50 ----
    LIBRISPEECH_SUBSETS  = ["dev-clean", "dev-other", "test-other"],
    LIBRISPEECH_MIRRORS  = ["https://openslr.trmal.net/resources/12", "https://openslr.elda.org/resources/12"],
    MAX_UTTS_PER_SPEAKER = 60, LIBRISPEECH_LOCAL_DIR = None, LS_VAL_FRAC = 0.15, LS_TEST_FRAC = 0.15,
    USE_ESC50 = True, ESC50_URL = "https://github.com/karoldvl/ESC-50/archive/master.zip", ESC50_MAX_CLIPS = 800, ESC50_LOCAL_DIR = None,
    SEED = 1023,
    # ---- storage ----
    DRIVE_DIR = "/content/drive/MyDrive/CAC_deepfake_detector", LOCAL_DIR = "/content/work",
    ORIG_CACHE = "cache_en", EXTRA_REAL_CACHE = "cache_extra_real_v3", RUN_TAG = "posttrain_v4",
    # ---- quick end-to-end check on CPU (tiny data, a few steps). Leave False for real training. ----
    SMOKE_TEST = False,
)
import os, json
if os.environ.get("CAC_NB_OVERRIDES"):              # used only by the offline self-test of this notebook
    CFG.update(json.load(open(os.environ["CAC_NB_OVERRIDES"])))
assert "test-clean" not in CFG["LIBRISPEECH_SUBSETS"], "test-clean holds the benchmark's human controls"
print(json.dumps(CFG, indent=1, default=str))

## 2. GPU, Drive, and automatic batch/precision settings
Each step runs the student on 1.5× the batch (noisy twins) plus a frozen V3 teacher, so batches are smaller than V3's.

In [ ]:
import sys, subprocess, torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
elif not os.environ.get("CAC_NB_OVERRIDES"):
    CFG["DRIVE_DIR"] = os.path.abspath("drive_out"); CFG["LOCAL_DIR"] = os.path.abspath("work")
os.makedirs(CFG["DRIVE_DIR"], exist_ok=True); os.makedirs(CFG["LOCAL_DIR"], exist_ok=True)

if torch.cuda.is_available():
    DEVICE = "cuda"
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    try:    USE_BF16 = torch.cuda.is_bf16_supported(including_emulation=False)
    except TypeError: USE_BF16 = torch.cuda.get_device_capability()[0] >= 8
    AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
    if   gpu_mem >= 70: BATCH, ACCUM = 24, 1
    elif gpu_mem >= 35: BATCH, ACCUM = 16, 2
    elif gpu_mem >= 20: BATCH, ACCUM = 8, 4
    else:               BATCH, ACCUM = 4, 8
    print(f"GPU: {gpu_name} ({gpu_mem:.0f} GB) | precision: {'bf16' if USE_BF16 else 'fp16'} | batch {BATCH} x accum {ACCUM}")
else:
    DEVICE, AMP_DTYPE, BATCH, ACCUM, gpu_mem, gpu_name = "cpu", None, 2, 1, 0, "cpu"
    print("⚠️ No GPU. Data preparation is fine on CPU; training refuses to run on CPU unless SMOKE_TEST=True.")
NUM_WORKERS = min(4, os.cpu_count() or 2) if not CFG["SMOKE_TEST"] else 0
print("CPU workers:", NUM_WORKERS)

## 3. Packages

In [ ]:
if IN_COLAB:
    %pip install -q soundfile scikit-learn matplotlib tqdm openpyxl
import transformers, numpy as np
print("torch", torch.__version__, "| transformers", transformers.__version__, "| numpy", np.__version__)
def _ffmpeg_has(encoder):
    return subprocess.run(f"ffmpeg -hide_banner -encoders 2>/dev/null | grep -q {encoder}", shell=True).returncode == 0
HAS_OPUS, HAS_MP3 = _ffmpeg_has("libopus"), _ffmpeg_has("libmp3lame")
print("ffmpeg libopus:", HAS_OPUS, "| libmp3lame:", HAS_MP3)

## 4. Imports, seeds, folders

In [ ]:
import re, io, math, time, glob, copy, random, shutil, tempfile, hashlib, tarfile, zipfile, urllib.request
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import pandas as pd, soundfile as sf
from scipy.signal import resample_poly, butter, sosfilt, lfilter
from tqdm.auto import tqdm
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import roc_curve, roc_auc_score

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(CFG["SEED"])

SR = CFG["SR"]; WIN = int(CFG["TRAIN_SEC"] * SR)
DRIVE = Path(CFG["DRIVE_DIR"]); LOCAL = Path(CFG["LOCAL_DIR"])
sfx = "_smoke" if CFG["SMOKE_TEST"] else ""
ORIG_CACHE = DRIVE / CFG["ORIG_CACHE"]
EXTRA_CACHE = DRIVE / CFG["EXTRA_REAL_CACHE"]                      # V3's cache is reused as-is when present
if not (EXTRA_CACHE / "manifest.csv").exists(): EXTRA_CACHE = DRIVE / f"{CFG['EXTRA_REAL_CACHE']}{sfx}"
COMM_CACHE = DRIVE / f"cache_commercial_v4{sfx}"
CKPT_DIR = DRIVE / f"ckpt_{CFG['RUN_TAG']}{sfx}"
EXPORT = DRIVE / f"export_{CFG['RUN_TAG']}{sfx}"
for d in (EXTRA_CACHE, COMM_CACHE, CKPT_DIR, LOCAL): d.mkdir(parents=True, exist_ok=True)
def on_drive(p): return Path(p) if Path(p).is_absolute() else DRIVE / p
AUDIO_EXT = (".wav", ".flac", ".mp3", ".m4a", ".ogg", ".aac")

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()

def download(urls, dest, desc):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0: return dest
    tmp = dest.with_suffix(dest.suffix + ".part"); last = None
    for url in ([urls] if isinstance(urls, str) else urls):
        for attempt in range(3):
            try:
                with urllib.request.urlopen(url, timeout=60) as r, open(tmp, "wb") as f, \
                     tqdm(total=int(r.headers.get("Content-Length") or 0), unit="B", unit_scale=True, desc=desc, mininterval=2) as bar:
                    for chunk in iter(lambda: r.read(1 << 20), b""):
                        f.write(chunk); bar.update(len(chunk))
                tmp.rename(dest); return dest
            except Exception as e:
                last = e; print(f"  {url}: {type(e).__name__}: {str(e)[:120]} (attempt {attempt+1})"); time.sleep(3 * (attempt + 1))
    raise RuntimeError(f"Could not download {desc}: {last}")

def save_json(path, obj): Path(path).write_text(json.dumps(obj, indent=1, default=float))

## 5. Load the V2 cache (MLAAD fakes + M-AILABS real), reused, not rebuilt
If this fails, run sections 1–6 of the V2 notebook once.

In [ ]:
if not ((ORIG_CACHE / "manifest.csv").exists() and (ORIG_CACHE / "audio_int16.npy").exists()):
    raise SystemExit(f"V2 cache not found in {ORIG_CACHE}. Run the V2 notebook's sections 1-6 first (or set ORIG_CACHE).")
orig_blob_local = LOCAL / "orig_audio_int16.npy"
if not orig_blob_local.exists():
    print("copying V2 cache to local disk (faster reads during training)...")
    shutil.copy(ORIG_CACHE / "audio_int16.npy", orig_blob_local)
BLOBS = [np.load(orig_blob_local, mmap_mode="r")]
orig = pd.read_csv(ORIG_CACHE / "manifest.csv")
orig["blob"] = 0
orig["source"] = np.where(orig.label == 1, "tts:" + orig.system.astype(str), "mailabs:" + orig.group.astype(str))
orig["corpus"] = np.where(orig.label == 1, "MLAAD", "M-AILABS")
print(orig.groupby(["split", "label"]).size().unstack().rename(columns={0: "real", 1: "fake"}))

## 6. LibriSpeech real speech (split by speaker), reused from V3
Built again only if `cache_extra_real_v3` is missing, using the same steps and seed as V3. `test-clean` is never downloaded; it holds the benchmark's human controls.

In [ ]:
def load_audio(path, sr=SR):
    try:
        x, s = sf.read(str(path), dtype="float32", always_2d=True)
    except Exception:
        r = subprocess.run(["ffmpeg", "-loglevel", "quiet", "-i", str(path), "-f", "f32le", "-ac", "1", "-ar", str(sr), "-"],
                           capture_output=True, check=True)
        return np.frombuffer(r.stdout, dtype=np.float32).copy()
    x = x.mean(axis=1)
    if s != sr:
        g = math.gcd(int(s), sr); x = resample_poly(x, sr // g, int(s) // g).astype(np.float32)
    return x

def trim_silence(x, sr=SR, top_db=40.0, frame=0.02):   # identical to V2/V3
    n = int(sr * frame)
    if len(x) < n * 2: return x
    k = len(x) // n
    e = 10 * np.log10(np.mean(x[: k * n].reshape(k, n) ** 2, axis=1) + 1e-12)
    keep = np.where(e > e.max() - top_db)[0]
    if len(keep) == 0: return x
    return x[keep[0] * n : (keep[-1] + 1) * n]

def to_int16(x):
    peak = np.abs(x).max()
    return None if peak < 1e-4 else (x / peak * 0.9 * 32767).astype(np.int16)

def preprocess_clip(path):                            # identical to V2's proc()
    try:
        x = trim_silence(load_audio(path))
        if len(x) < CFG["MIN_SEC"] * SR: return None
        return to_int16(x[: int(CFG["MAX_STORE_SEC"] * SR)])
    except Exception:
        return None

def write_cache(arrays, items, cache_dir, desc):
    lengths = np.array([len(a) for a in arrays], np.int64)
    out = np.lib.format.open_memmap(LOCAL / f"{desc}.npy", mode="w+", dtype=np.int16, shape=(int(lengths.sum()),))
    off = 0
    for a in arrays: out[off:off + len(a)] = a; off += len(a)
    out.flush(); del out
    m = pd.DataFrame(items); m["offset"] = np.concatenate([[0], np.cumsum(lengths)[:-1]]); m["length"] = lengths
    shutil.copy(LOCAL / f"{desc}.npy", cache_dir / "audio_int16.npy"); m.to_csv(cache_dir / "manifest.csv", index=False)
    return m

BENCHMARK_SPEAKERS = {"1320", "6930"}
if (EXTRA_CACHE / "manifest.csv").exists() and (EXTRA_CACHE / "audio_int16.npy").exists():
    extra = pd.read_csv(EXTRA_CACHE / "manifest.csv", dtype={"speaker": str})
    print("LibriSpeech cache found on Drive:", len(extra), "clips")
else:
    print("LibriSpeech cache missing: rebuilding it exactly as V3 did (about 1 GB download)")
    root = Path(CFG["LIBRISPEECH_LOCAL_DIR"]) if CFG["LIBRISPEECH_LOCAL_DIR"] else LOCAL / "librispeech"
    root.mkdir(parents=True, exist_ok=True)
    for subset in CFG["LIBRISPEECH_SUBSETS"]:
        if (root / "LibriSpeech" / subset).exists(): continue
        tgz = download([f"{m}/{subset}.tar.gz" for m in CFG["LIBRISPEECH_MIRRORS"]], LOCAL / f"{subset}.tar.gz", subset)
        with tarfile.open(tgz) as t:
            t.extractall(root, filter="data") if hasattr(tarfile, "data_filter") else t.extractall(root)
        tgz.unlink()
    rng = random.Random(1022); items = []                     # V3's seed, so the speaker split matches V3
    for subset in CFG["LIBRISPEECH_SUBSETS"]:
        for spk_dir in sorted((root / "LibriSpeech" / subset).iterdir()):
            if not spk_dir.is_dir(): continue
            assert spk_dir.name not in BENCHMARK_SPEAKERS, "benchmark speaker found in training data"
            files = sorted(spk_dir.rglob("*.flac"))
            for p in rng.sample(files, min(len(files), CFG["MAX_UTTS_PER_SPEAKER"] if not CFG["SMOKE_TEST"] else 5)):
                items.append(dict(path=str(p), subset=subset, speaker=spk_dir.name))
    arrays, kept = [], []
    with ThreadPoolExecutor(NUM_WORKERS * 2 or 2) as ex:
        for it, a in zip(items, tqdm(ex.map(lambda it: preprocess_clip(it["path"]), items), total=len(items), desc="librispeech")):
            if a is not None: arrays.append(a); kept.append(it)
    extra = write_cache(arrays, kept, EXTRA_CACHE, "librispeech")
    extra["speaker"] = extra["speaker"].astype(str)
    speakers = sorted(extra.speaker.unique()); random.Random(1022).shuffle(speakers)
    n_te, n_va = round(len(speakers) * CFG["LS_TEST_FRAC"]), round(len(speakers) * CFG["LS_VAL_FRAC"])
    te, va = set(speakers[:n_te]), set(speakers[n_te:n_te + n_va])
    extra["split"] = np.where(extra.speaker.isin(te), "test", np.where(extra.speaker.isin(va), "val", "train"))
    extra.to_csv(EXTRA_CACHE / "manifest.csv", index=False)

extra_blob_local = LOCAL / "librispeech.npy"
if not extra_blob_local.exists(): shutil.copy(EXTRA_CACHE / "audio_int16.npy", extra_blob_local)
BLOBS.append(np.load(extra_blob_local, mmap_mode="r"))
extra = extra.assign(label=0, system="bonafide_librispeech", group="ls:" + extra.speaker.astype(str), blob=1,
                     source="librispeech:" + extra.speaker.astype(str), corpus="LibriSpeech")
COLS = ["path", "label", "system", "group", "offset", "length", "split", "blob", "source", "corpus"]
assert not set(extra.speaker.astype(str)) & BENCHMARK_SPEAKERS
print(extra.groupby("split").speaker.nunique().rename("LibriSpeech speakers"))

## 7. ElevenLabs and other commercial voice-clone audio (new in V4)
Collects ElevenLabs/Respeecher clips, splits them **by voice**, and caches 8-second segments to Drive.

The data mode is chosen automatically:
- **external:** the Mendeley set has at least `MIN_EXTERNAL_ELEVEN` ElevenLabs clips. Your 3 benchmark voices become *test only*.
- **own_voices_fallback:** only your clips. `FALLBACK_TEST_VOICES` are test only; the other voices train, with their last 20% kept for validation.

Every file's SHA-256 is recorded, so the benchmark table later knows exactly which recordings were seen in training.

In [ ]:
OWN_EL_DIR, EXT_DIR = DRIVE / CFG["ELEVEN_OWN_SUBDIR"], DRIVE / CFG["ELEVEN_EXTERNAL_SUBDIR"]
BENCH_DIR, OWN_REAL_DIR = DRIVE / CFG["BENCHMARK_SUBDIR"], DRIVE / CFG["OWN_REAL_SUBDIR"]

def audio_files(d):
    d = Path(d)
    return sorted(p for p in d.rglob("*") if p.suffix.lower() in AUDIO_EXT and not p.name.startswith(".")) if d.exists() else []

def voice_of(p, root):
    m = re.search(r"_([A-Za-z]+)_ivc", p.name)                 # ElevenLabs export names, e.g. ..._shrey_ivc_...
    if m: return m.group(1).lower()
    rel = p.relative_to(root)
    return rel.parts[0].lower() if len(rel.parts) > 1 else re.sub(r"[^a-z0-9]+", "_", p.stem.lower())

def external_items():
    files = audio_files(EXT_DIR)
    if not files: return []
    stems = {p.stem.lower(): p for p in files}
    meta = {}
    for m in sorted(list(EXT_DIR.rglob("*.xlsx")) + list(EXT_DIR.rglob("*.xls")) + list(EXT_DIR.rglob("*.csv"))):
        try: d = pd.read_csv(m) if m.suffix.lower() == ".csv" else pd.read_excel(m)
        except Exception as e: print("  could not read", m.name, e); continue
        best, hits = None, 0
        for c in d.columns:                                   # the column whose values match the audio file names
            h = int(d[c].astype(str).str.strip().map(lambda s: Path(s).stem.lower()).isin(stems).sum())
            if h > hits: best, hits = c, h
        if not hits: continue
        print(f"  metadata {m.name}: column '{best}' matches {hits}/{len(d)} rows")
        for _, r in d.iterrows():
            key = Path(str(r[best]).strip()).stem.lower()
            if key in stems: meta[key] = {str(k).strip().lower(): str(v) for k, v in r.items()}
    items = []
    for p in files:
        row = meta.get(p.stem.lower(), {})
        text = (" ".join(row.values()) + " " + str(p.relative_to(EXT_DIR))).lower()
        tool = "elevenlabs" if "eleven" in text else "respeecher" if "respeecher" in text else None
        if tool is None or (tool == "respeecher" and not CFG["USE_RESPEECHER"]): continue
        mode = ("voice-to-voice" if re.search(r"voice.?to.?voice|v2v|speech.?to.?speech", text) else
                "tts" if re.search(r"text.?to.?speech|\btts\b", text) else "unknown")
        spk = next((k for k in row if re.search(r"speaker|voice.?id|locutor|person|subject", k)), None)
        items.append(dict(path=str(p), tool=tool, mode=mode, origin="external", label=1,
                          voice=f"{tool}:{row[spk]}" if spk else f"{tool}:file:{p.stem.lower()}", voice_known=spk is not None))
    return items

ext_items = external_items()
n_ext_el = sum(i["tool"] == "elevenlabs" for i in ext_items)
MODE = "external" if n_ext_el >= CFG["MIN_EXTERNAL_ELEVEN"] else "own_voices_fallback"

own_items, seen_sha = [], set()
for root, origin in ((OWN_EL_DIR, "own"), (BENCH_DIR, "benchmark")):
    for p in audio_files(root):
        if origin == "benchmark" and "elevenlabs" not in p.name.lower(): continue   # benchmark humans are test-only controls
        h = sha256(p)
        if h in seen_sha: continue
        seen_sha.add(h)
        own_items.append(dict(path=str(p), tool="elevenlabs", mode="unknown", origin=origin, label=1, voice=voice_of(p, root), voice_known=True))
real_items = [dict(path=str(p), tool="own_real", mode="human", origin="own_real", label=0, voice="own:" + voice_of(p, OWN_REAL_DIR), voice_known=True)
              for p in audio_files(OWN_REAL_DIR)]
if not ext_items and not own_items:
    MODE = "none"
print(f"DATA MODE: {MODE} | external ElevenLabs {n_ext_el}, external Respeecher {len(ext_items) - n_ext_el}, "
      f"own/benchmark ElevenLabs files {len(own_items)}, own real files {len(real_items)}")
if MODE == "external" and not all(i["voice_known"] for i in ext_items if i["tool"] == "elevenlabs"):
    print("⚠️ The external metadata has no speaker/voice column: it is split by FILE, so one voice may appear in "
          "train and val. Validation scores on the external set may be optimistic; the held-out benchmark voices are not affected.")
if MODE == "external":
    print("ℹ️ The external set's language isn't stated on its page. Compare the external-test and benchmark rows at the end "
          "to see whether it transfers to English ElevenLabs.")
if MODE == "own_voices_fallback":
    print(f"⚠️ Own-voices fallback: only {len({i['voice'] for i in own_items})} ElevenLabs voices. Voices {CFG['FALLBACK_TEST_VOICES']} are test-only; "
          "the others are trained on, so the repo benchmark is no longer independent for them.")
if MODE == "none":
    print("⚠️ No ElevenLabs audio found. V4 will still train the robustness fixes, but the ElevenLabs objective falls back to all fakes.")

def segment(path):
    x = trim_silence(load_audio(path)); L, out = int(CFG["SEGMENT_SEC"] * SR), []
    for s in range(0, len(x), L):
        seg = x[s:s + L]
        if len(seg) >= CFG["MIN_SEC"] * SR and (a := to_int16(seg)) is not None: out.append(a)
    return out

all_items = ext_items + own_items + real_items
for it in all_items: it["file_sha256"] = sha256(it["path"]) if "file_sha256" not in it else it["file_sha256"]
signature = hashlib.sha256(json.dumps(sorted((i["file_sha256"], i["voice"], i["tool"]) for i in all_items)
                                      + [MODE, CFG["SEGMENT_SEC"], CFG["FALLBACK_TEST_VOICES"], CFG["BENCHMARK_VOICES"],
                                         CFG["ELEVEN_VAL_FRAC"], CFG["ELEVEN_TEST_FRAC"], CFG["SEED"]]).encode()).hexdigest()

def assign_splits(m):
    m["split"] = "train"; rng = random.Random(CFG["SEED"])
    ext = m.origin == "external"
    for tool in m[ext].system.unique():                       # external: by voice group
        groups = sorted(m[ext & (m.system == tool)].group.unique()); rng.shuffle(groups)
        n_te, n_va = round(len(groups) * CFG["ELEVEN_TEST_FRAC"]), round(len(groups) * CFG["ELEVEN_VAL_FRAC"])
        m.loc[ext & m.group.isin(groups[:n_te]), "split"] = "test"
        m.loc[ext & m.group.isin(groups[n_te:n_te + n_va]), "split"] = "val"
    own = m.origin.isin(["own", "benchmark"])
    test_voices = set(CFG["BENCHMARK_VOICES"]) if MODE == "external" else set(CFG["FALLBACK_TEST_VOICES"])
    m.loc[own & m.group.isin(test_voices), "split"] = "test"
    for (path,), d in m[own & ~m.group.isin(test_voices)].groupby(["path"]):        # time split inside each trained file
        k = int(math.ceil(len(d) * CFG["ELEVEN_VAL_FRAC"])) if len(d) > 1 else 0
        if k: m.loc[d.sort_values("seg").index[-k:], "split"] = "val"
    real = m.origin == "own_real"
    spk = sorted(m[real].group.unique()); rng.shuffle(spk)
    if len(spk) >= 3:
        m.loc[real & (m.group == spk[0]), "split"] = "test"; m.loc[real & (m.group == spk[1]), "split"] = "val"
    else:
        for (path,), d in m[real].groupby(["path"]):
            k = int(math.ceil(len(d) * 0.2)) if len(d) > 1 else 0
            if k: m.loc[d.sort_values("seg").index[-k:], "split"] = "val"
        if len(spk): print("⚠️ fewer than 3 own real speakers: own real val is a time split of the same speakers")
    return m

sig_file = COMM_CACHE / "signature.txt"
if all_items and sig_file.exists() and sig_file.read_text() == signature and (COMM_CACHE / "manifest.csv").exists():
    comm = pd.read_csv(COMM_CACHE / "manifest.csv"); print("commercial cache found on Drive:", len(comm), "segments")
elif all_items:
    arrays, rows = [], []
    for it in tqdm(all_items, desc="segmenting"):
        try: segs = segment(it["path"])
        except Exception as e: print("  skipped", Path(it["path"]).name, type(e).__name__); continue
        for j, a in enumerate(segs):
            arrays.append(a)
            rows.append(dict(path=it["path"], label=it["label"], system=it["tool"], group=it["voice"], origin=it["origin"],
                             mode=it["mode"], seg=j, file_sha256=it["file_sha256"]))
    if not arrays: raise RuntimeError("Commercial/own audio found but none could be decoded")
    comm = assign_splits(write_cache(arrays, rows, COMM_CACHE, "commercial"))
    comm.to_csv(COMM_CACHE / "manifest.csv", index=False); sig_file.write_text(signature)
else:
    comm = pd.DataFrame(columns=["path", "label", "system", "group", "origin", "mode", "seg", "file_sha256", "offset", "length", "split"])

if len(comm):
    comm_blob_local = LOCAL / "commercial.npy"
    shutil.copy(COMM_CACHE / "audio_int16.npy", comm_blob_local)
    BLOBS.append(np.load(comm_blob_local, mmap_mode="r"))
    comm = comm.assign(blob=len(BLOBS) - 1, source=comm.origin + ":" + comm.group.astype(str),
                       corpus=np.select([comm.system == "elevenlabs", comm.system == "respeecher"], ["ElevenLabs", "Respeecher"], "OwnReal"))
    for s in ("train", "val"):                 # no test voice/file may reach training or validation
        assert not (set(comm.loc[comm.split == "test", "group"]) & set(comm.loc[comm.split == s, "group"])), f"test voice leaked into {s}"
    print(comm.groupby(["corpus", "origin", "split"]).agg(segments=("seg", "size"), voices=("group", "nunique"),
                                                          minutes=("length", lambda v: round(v.sum() / SR / 60, 1))).to_string())
TRAINED_FILE_SHA = set(comm.loc[comm.split.isin(["train", "val"]), "file_sha256"]) if len(comm) else set()
TRAINED_VOICES = sorted(set(comm.loc[comm.split.isin(["train", "val"]) & (comm.label == 1), "group"])) if len(comm) else []
print("ElevenLabs/commercial voices used in training or validation:", len(TRAINED_VOICES))

EXTRA_COLS = ["origin", "mode", "file_sha256"]
manifest = pd.concat([orig[COLS], extra[COLS], comm[COLS + EXTRA_COLS] if len(comm) else None], ignore_index=True)
for c in EXTRA_COLS: manifest[c] = manifest[c].fillna("") if c in manifest else ""
for s in ("train", "val", "test"):          # LibriSpeech speakers never cross splits
    others = set(extra.loc[extra.split != s, "speaker"]); assert not (set(extra.loc[extra.split == s, "speaker"]) & others)
print(manifest.groupby(["split", "corpus"]).size().unstack(fill_value=0))

## 8. Background-sound banks: recorded (ESC-50), synthetic music, DTMF, babble (as V3)

In [ ]:
def esc50_bank():
    if not CFG["USE_ESC50"]: return []
    base = Path(CFG["ESC50_LOCAL_DIR"]) if CFG["ESC50_LOCAL_DIR"] else LOCAL / "esc50"
    audio_dir = next(iter(base.glob("*/audio")), None) if base.exists() else None
    if audio_dir is None:
        z = download(CFG["ESC50_URL"], LOCAL / "esc50.zip", "ESC-50")
        with zipfile.ZipFile(z) as f: f.extractall(base)
        z.unlink(); audio_dir = next(iter(base.glob("*/audio")))
    files = sorted(audio_dir.glob("*.wav")); random.Random(CFG["SEED"]).shuffle(files)
    out = []
    for p in tqdm(files[: CFG["ESC50_MAX_CLIPS"] if not CFG["SMOKE_TEST"] else 20], desc="ESC-50"):
        x = load_audio(str(p))
        if np.abs(x).max() > 1e-3: out.append((x / (np.abs(x).max() + 1e-9) * 0.9 * 32767).astype(np.int16))
    return out
try:
    NOISE_BANK = esc50_bank()
except Exception as e:
    NOISE_BANK = []; print("⚠️ ESC-50 unavailable, using synthetic noise only:", e)
print("recorded noise clips:", len(NOISE_BANK))

def synth_music(n, rng):
    t = np.arange(n) / SR
    if rng.random() < 0.5:
        root = rng.uniform(110, 330); ratios = [1, 1.26, 1.5, 2][: rng.integers(2, 5)]
        x = sum(np.sin(2 * np.pi * root * r * t + rng.uniform(0, 6.3)) for r in ratios)
        x = x * (0.6 + 0.4 * np.sin(2 * np.pi * rng.uniform(0.5, 4) * t))
    else:
        note = int(SR * rng.uniform(0.15, 0.5)); x = np.zeros(n)
        for s in range(0, n, note):
            f = 220 * 2 ** (rng.integers(0, 15) / 12); m = min(note, n - s); tt = np.arange(m) / SR
            x[s:s + m] = sum(np.sin(2 * np.pi * k * f * tt) / k for k in range(1, 6)) * np.exp(-tt * rng.uniform(1, 6))
    return x.astype(np.float32)

DTMF = [(697, 1209), (697, 1336), (697, 1477), (770, 1209), (770, 1336), (770, 1477), (852, 1209), (852, 1336), (852, 1477), (941, 1336)]
def synth_dtmf(n, rng):
    t = np.arange(n) / SR; x = np.zeros(n); s = 0
    while s < n:
        on, off = int(SR * rng.uniform(.08, .5)), int(SR * rng.uniform(.05, .4)); a, b = DTMF[rng.integers(len(DTMF))]
        e = min(n, s + on); x[s:e] = np.sin(2 * np.pi * a * t[s:e]) + np.sin(2 * np.pi * b * t[s:e]); s = e + off
    return x.astype(np.float32)

def colored_noise(n, rng, kind=None):
    w = rng.standard_normal(n).astype(np.float32); kind = kind or rng.choice(["white", "pink", "brown"])
    if kind == "brown": w = np.cumsum(w); w -= np.linspace(w[0], w[-1], n)
    elif kind == "pink": w = lfilter([0.049922035, -0.095993537, 0.050612699, -0.004408786], [1, -2.494956002, 2.017265875, -0.522189400], w)
    return (w / (w.std() + 1e-9)).astype(np.float32)

def recorded_noise(n, rng):
    if not NOISE_BANK: return colored_noise(n, rng)
    x = NOISE_BANK[rng.integers(len(NOISE_BANK))].astype(np.float32) / 32768
    if len(x) < n: x = np.tile(x, int(np.ceil(n / len(x))))
    s = rng.integers(0, len(x) - n + 1); return x[s:s + n]

TRAIN_REAL_IDX = np.flatnonzero(((manifest.split == "train") & (manifest.label == 0)).to_numpy())   # babble: training real speech only
def babble(n, rng):
    x = np.zeros(n, np.float32)
    for _ in range(rng.integers(2, 5)):
        r = manifest.iloc[TRAIN_REAL_IDX[rng.integers(len(TRAIN_REAL_IDX))]]
        a = np.asarray(BLOBS[int(r.blob)][int(r.offset):int(r.offset) + int(r.length)], np.float32) / 32768
        x += fit_window_n(a, n, rng)
    return x

def fit_window_n(x, n, rng=None, start=None):
    if start is not None: x = x[start:start + n]
    if len(x) < n: x = np.tile(x, int(np.ceil(n / max(len(x), 1))))[:n] if len(x) else np.zeros(n, np.float32)
    elif len(x) > n:
        s = rng.integers(0, len(x) - n + 1) if rng is not None else (len(x) - n) // 2; x = x[s:s + n]
    return x.astype(np.float32)
print("banks ready")

## 9. Augmentation (label-blind) and evaluation conditions
`train_augment` never receives the label. Knob-controlled probabilities come from the Guardian's knob dict `K`, which is passed in when each loader is built.

New in V4:
- wideband MP3 and band-limit round trips (both classes)
- `noisy_twin` for the consistency loss
- the benchmark's own conditions (`BENCH_CONDITIONS`), so the ElevenLabs objective is measured the way the repo benchmark measures it

In [ ]:
_SOS_PHONE = butter(4, [300, 3400], btype="bandpass", fs=8000, output="sos")

def mulaw_roundtrip(x, mu=255):
    x = np.clip(x, -1, 1); y = np.sign(x) * np.log1p(mu * np.abs(x)) / np.log1p(mu)
    q = np.round((y + 1) / 2 * mu) / mu * 2 - 1
    return (np.sign(q) * ((1 + mu) ** np.abs(q) - 1) / mu).astype(np.float32)

def codec_roundtrip(x, codec, bitrate, sr=8000):
    with tempfile.TemporaryDirectory() as d:
        a, b, c = f"{d}/a.wav", f"{d}/b.{'ogg' if codec == 'opus' else 'mp3'}", f"{d}/c.wav"
        sf.write(a, x, sr)
        enc = ["-c:a", "libopus"] if codec == "opus" else ["-c:a", "libmp3lame"]
        r1 = subprocess.run(["ffmpeg", "-loglevel", "quiet", "-y", "-i", a, *enc, "-b:a", bitrate, b])
        r2 = subprocess.run(["ffmpeg", "-loglevel", "quiet", "-y", "-i", b, "-ar", str(sr), "-ac", "1", c]) if r1.returncode == 0 else r1
        if r2.returncode != 0: return x
        y, _ = sf.read(c, dtype="float32"); y = y[: len(x)]
        return (np.pad(y, (0, len(x) - len(y))) if len(y) < len(x) else y).astype(np.float32)

def phone_channel(x, rng):
    n = len(x); x = x * 10 ** (rng.uniform(-12, 3) / 20)
    y = resample_poly(x, 1, 2).astype(np.float32)
    if rng.random() < 0.8: y = sosfilt(_SOS_PHONE, y).astype(np.float32)
    u = rng.random()
    if HAS_OPUS and u < CFG["OPUS_P"]: y = codec_roundtrip(y, "opus", rng.choice(["8k", "12k", "16k"]))
    elif HAS_MP3 and u < CFG["OPUS_P"] + CFG["MP3_P"]: y = codec_roundtrip(y, "mp3", rng.choice(["16k", "24k", "32k"]))
    if rng.random() < 0.85: y = mulaw_roundtrip(y)
    if rng.random() < 0.3:
        fl = 160; k = len(y) // fl; y[: k * fl][np.repeat(rng.random(k) < 0.02, fl)] = 0
    y = resample_poly(y, 2, 1).astype(np.float32)[:n]
    return np.pad(y, (0, n - len(y))) if len(y) < n else y

def band8k(x): return resample_poly(resample_poly(x, 1, 2), 2, 1)[:len(x)].astype(np.float32)

def source_roundtrip(x, rng):
    """New in V4: imitate how clips of either class may have been stored (wideband MP3, lower sample rates)."""
    if HAS_MP3 and rng.random() < CFG["WB_MP3_P"]: x = codec_roundtrip(x, "mp3", rng.choice(["32k", "48k", "64k"]), sr=SR)
    if rng.random() < CFG["BANDLIMIT_P"]:
        x = band8k(x) if rng.random() < .5 else resample_poly(resample_poly(x, 3, 4), 4, 3)[:len(x)].astype(np.float32)
    return x

def mix_at_snr(x, noise, snr_db):
    p = np.mean(x ** 2) + 1e-10; q = np.mean(noise ** 2) + 1e-10
    return (x + noise * np.sqrt(p / q / 10 ** (snr_db / 10))).astype(np.float32)

def background(n, rng):
    u = rng.random()
    return synth_music(n, rng) if u < .35 else synth_dtmf(n, rng) if u < .5 else babble(n, rng) if u < .75 else recorded_noise(n, rng)

def insert_gaps(x, rng, digital=None):
    x = x.copy(); n = len(x)
    for _ in range(rng.integers(1, 3)):
        L = min(n - 1, int(SR * rng.uniform(*CFG["GAP_SEC"])))
        s = rng.choice([0, n - L, rng.integers(0, n - L + 1)])
        dig = (rng.random() < CFG["GAP_DIGITAL_FRAC"]) if digital is None else digital
        x[s:s + L] = 0 if dig else colored_noise(L, rng) * 10 ** (rng.uniform(*CFG["NOISE_FLOOR_DBFS"]) / 20)
    return x

def noise_gate(x, rng):
    fl = int(SR * .02); k = len(x) // fl
    if k == 0: return x
    e = np.sqrt(np.mean(x[: k * fl].reshape(k, fl) ** 2, 1) + 1e-12)
    thr = np.percentile(e, rng.uniform(20, 50))
    x = x.copy(); x[: k * fl][np.repeat(e <= thr, fl)] = 0; return x

def train_noise(n, rng):
    u = rng.random()
    if u < CFG["RECORDED_NOISE_FRAC"]: return recorded_noise(n, rng)
    if u < CFG["RECORDED_NOISE_FRAC"] + CFG["WHITE_NOISE_FRAC"]: return colored_noise(n, rng, "white")
    return colored_noise(n, rng)

def train_augment(x, rng, K):
    # Identical for real and fake clips: it has no access to the label.
    n = len(x)
    x = source_roundtrip(x, rng)
    if rng.random() < K["OVERLAY_P"]: x = mix_at_snr(x, background(n, rng), rng.uniform(*CFG["OVERLAY_SNR_DB"]))
    if rng.random() < CFG["PHONE_AUG_P"]: x = phone_channel(x, rng)
    if rng.random() < K["NOISE_P"]: x = mix_at_snr(x, train_noise(n, rng), rng.uniform(*CFG["NOISE_SNR_DB"]))
    if rng.random() < CFG["NOISE_FLOOR_P"]: x = x + colored_noise(n, rng) * 10 ** (rng.uniform(*CFG["NOISE_FLOOR_DBFS"]) / 20)
    if rng.random() < CFG["NOISE_GATE_P"]: x = noise_gate(x, rng)
    if rng.random() < K["GAP_P"]: x = insert_gaps(x, rng)
    if rng.random() < CFG["CLIP_P"]:
        c = np.abs(x).max() * rng.uniform(.3, .7); x = np.clip(x, -c, c)
    return np.clip(x, -1, 1).astype(np.float32)

def noisy_twin(x, rng):
    """A noisier copy of an already-augmented window, for the consistency loss."""
    noise = colored_noise(len(x), rng, "white" if rng.random() < .7 else "pink")
    return np.clip(mix_at_snr(x, noise, rng.uniform(*CFG["CONSIST_SNR_DB"])), -1, 1).astype(np.float32)

def nonspeech_window(n, rng):
    u = rng.random()
    x = synth_music(n, rng) if u < .25 else synth_dtmf(n, rng) if u < .45 else recorded_noise(n, rng) if u < .75 else colored_noise(n, rng)
    x = x / (np.abs(x).max() + 1e-9) * rng.uniform(.01, .9)
    return insert_gaps(x, rng) if rng.random() < .3 else x

# V3's robustness conditions (validation/test), plus the repo benchmark's conditions (ElevenLabs objective, benchmark table).
EVAL_CONDITIONS = ["clean", "phone", "noise20", "noise10", "phone_noise10", "gap2s", "music10"]
BENCH_CONDITIONS = ["original", "8khz_bandwidth_only", "g711_mulaw", "low_volume_minus20dB", "mild_clipping",
                    "white_noise_30dB_SNR", "white_noise_20dB_SNR", "white_noise_10dB_SNR"]
def pcm16(x): return (np.clip(x, -1, 1) * 32767).astype(np.int16).astype(np.float32) / 32768

def g711(x):
    b = io.BytesIO(); sf.write(b, band8k(x)[::2], 8000, format="WAV", subtype="ULAW"); b.seek(0)
    y, s = sf.read(b, dtype="float32"); return resample_poly(y, 2, 1)[:len(x)].astype(np.float32)

def eval_condition(x, cond, rng):
    n = len(x)
    if cond == "clean": return x
    if cond == "phone": return np.clip(mix_at_snr(phone_channel(x, rng), colored_noise(n, rng, "white"), 25), -1, 1)
    if cond == "noise20": return mix_at_snr(x, colored_noise(n, rng, "white"), 20)
    if cond == "noise10": return mix_at_snr(x, colored_noise(n, rng, "white"), 10)
    if cond == "phone_noise10": return mix_at_snr(phone_channel(x, rng), colored_noise(n, rng, "white"), 10)
    if cond == "gap2s": y = x.copy(); y[: SR * 2] = 0; return y
    if cond == "music10": return mix_at_snr(x, synth_music(n, np.random.default_rng(7)), 10)
    if cond == "original": return pcm16(x)
    if cond == "8khz_bandwidth_only": return pcm16(band8k(x))
    if cond == "g711_mulaw": return pcm16(g711(x))
    if cond == "low_volume_minus20dB": return pcm16(x * .1)
    if cond == "mild_clipping": return pcm16(np.clip(x / (np.quantile(np.abs(x), .99) + 1e-9), -1, 1))
    if cond.startswith("white_noise_"):
        snr = int(cond.split("_")[2][:-2]); w = rng.standard_normal(n).astype(np.float32)
        return pcm16(x + w * np.sqrt(np.mean(x * x) / (10 ** (snr / 10) * np.mean(w * w))))
    raise ValueError(cond)

def normalize(x):                                         # same per-window z-normalization as inference
    return ((x - x.mean()) / (x.std() + 1e-7)).astype(np.float32)

K = {k: CFG[k] for k in ("NOISE_P", "GAP_P", "OVERLAY_P", "NONSPEECH_AS_REAL_P", "CONSIST_WEIGHT", "KD_WEIGHT",
                         "ANCHOR_RATE", "ELEVEN_FAKE_SHARE")}
K["LR_SCALE"] = 1.0
_t = np.random.default_rng(0).standard_normal(WIN).astype(np.float32) * .1
for c in EVAL_CONDITIONS + BENCH_CONDITIONS: assert eval_condition(_t, c, np.random.default_rng(1)).shape == (WIN,), c
assert train_augment(_t, np.random.default_rng(2), K).shape == (WIN,) and noisy_twin(_t, np.random.default_rng(3)).shape == (WIN,)
print("augmentation OK | initial knobs:", K)

## 10. Datasets and loaders
Training items return `(window, noisy_twin, label, kind)`:

| `kind` | Meaning |
|---:|---|
| 0 | MLAAD fake |
| 1 | commercial fake (ElevenLabs/Respeecher) |
| 2 | real speech |
| 3 | non-speech labeled not-AI |

The sampler gives the two classes equal weight. **Fakes:** `ELEVEN_FAKE_SHARE` ElevenLabs, `OTHER_COMMERCIAL_SHARE` Respeecher, the rest MLAAD. **Real:** `OWN_REAL_SHARE` own recordings, the rest split equally between M-AILABS and LibriSpeech.

The loader is rebuilt whenever the Guardian changes a knob, because worker processes keep their own copy of the knobs.

In [ ]:
COMMERCIAL = ["ElevenLabs", "Respeecher"]

class AudioDS(Dataset):
    def __init__(self, df, train=False, condition="clean", seed=0, knobs=None):
        self.df = df.reset_index(drop=True); self.train, self.cond, self.seed = train, condition, seed
        self.blob, self.off, self.len, self.lab = (self.df[c].to_numpy() for c in ["blob", "offset", "length", "label"])
        self.kind = np.where(self.lab == 1, np.where(self.df.corpus.isin(COMMERCIAL), 1, 0), 2)
        self.start = self.df["win_start"].to_numpy() if "win_start" in self.df else None
        self.K = dict(knobs or K); self.rng = np.random.default_rng(seed)
    def __len__(self): return len(self.df)
    def clip(self, i):
        o, n = int(self.off[i]), int(self.len[i])
        return np.asarray(BLOBS[int(self.blob[i])][o:o + n], dtype=np.float32) / 32768.0
    def __getitem__(self, i):
        if self.train:
            rng = self.rng
            if rng.random() < self.K["NONSPEECH_AS_REAL_P"]:
                x = torch.from_numpy(normalize(nonspeech_window(WIN, rng)))
                return x, x.clone(), torch.tensor(0.0), torch.tensor(3)
            x = train_augment(fit_window_n(self.clip(i), WIN, rng), rng, self.K)
            return (torch.from_numpy(normalize(x)), torch.from_numpy(normalize(noisy_twin(x, rng))),
                    torch.tensor(float(self.lab[i])), torch.tensor(int(self.kind[i])))
        rng = np.random.default_rng(self.seed * 100003 + i)
        start = None if self.start is None else int(self.start[i])
        x = eval_condition(fit_window_n(self.clip(i), WIN, None, start), self.cond, rng)
        return torch.from_numpy(normalize(x)), torch.tensor(float(self.lab[i]))

def worker_init(_):
    info = torch.utils.data.get_worker_info()
    info.dataset.rng = np.random.default_rng(torch.initial_seed() % 2**32)

def expand_windows(frame):
    """Every non-overlapping 4 s window of each clip (used for the short commercial segments)."""
    rows = []
    for r in frame.itertuples(index=False):
        n = int(r.length)
        for s in (range(0, n - WIN + 1, WIN) if n >= WIN else [None]):
            rows.append({**r._asdict(), "win_start": s if s is not None else 0})
    return pd.DataFrame(rows, columns=list(frame.columns) + ["win_start"])

df = manifest.copy()
if CFG["SMOKE_TEST"]:
    df = pd.concat([d.head(12 if c not in COMMERCIAL else 40) for (s, l, c), d in df.groupby(["split", "label", "corpus"])])
train_df, val_df, test_df = (df[df.split == s] for s in ["train", "val", "test"])

def sampler_weights(frame, K):
    lab, corp = frame.label.to_numpy(), frame.corpus.to_numpy(); w = np.zeros(len(frame))
    fake, real = lab == 1, lab == 0
    el, oc = fake & (corp == "ElevenLabs"), fake & (corp == "Respeecher"); ml = fake & ~el & ~oc
    s_el = K["ELEVEN_FAKE_SHARE"] if el.any() else 0.0; s_oc = CFG["OTHER_COMMERCIAL_SHARE"] if oc.any() else 0.0
    for m, s in ((el, s_el), (oc, s_oc), (ml, 1 - s_el - s_oc)):
        if m.any(): w[m] = 0.5 * s / m.sum()
    own = real & (corp == "OwnReal"); s_own = CFG["OWN_REAL_SHARE"] if own.any() else 0.0
    if own.any(): w[own] = 0.5 * s_own / own.sum()
    others = sorted(set(corp[real & ~own]))
    for c in others:
        m = real & (corp == c); w[m] = 0.5 * (1 - s_own) / len(others) / m.sum()
    return w

dl_kw = dict(num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"), worker_init_fn=worker_init if NUM_WORKERS else None,
             persistent_workers=NUM_WORKERS > 0)
def make_train_loader(K, seed):
    w = sampler_weights(train_df, K)
    sampler = WeightedRandomSampler(torch.tensor(w, dtype=torch.double), num_samples=len(train_df), replacement=True,
                                    generator=torch.Generator().manual_seed(int(seed)))
    return DataLoader(AudioDS(train_df, train=True, seed=seed, knobs=K), batch_size=BATCH, sampler=sampler, drop_last=True, **dl_kw)

def eval_loader(frame, cond, seed=1):
    return DataLoader(AudioDS(frame, condition=cond, seed=seed), batch_size=BATCH * 2,
                      num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))

# Validation used by the Guardian: a fixed, stratified subsample of the non-ElevenLabs validation clips,
# plus every 4 s window of the ElevenLabs validation segments under the benchmark's conditions.
def stratified(frame, cap, seed):
    if len(frame) <= cap: return frame
    return pd.concat([d.sample(max(1, round(cap * len(d) / len(frame))), random_state=seed)
                      for _, d in frame.groupby(["corpus", "label"])])
VAL_CORE_FULL = val_df[val_df.corpus != "ElevenLabs"]
VAL_CORE = stratified(VAL_CORE_FULL, CFG["VAL_CHECK_MAX_CLIPS"], CFG["SEED"])
EL_VAL, EL_TEST = val_df[val_df.corpus == "ElevenLabs"], test_df[test_df.corpus == "ElevenLabs"]
TEST_CORE = test_df[test_df.corpus != "ElevenLabs"]
w0 = sampler_weights(train_df, K)
print("expected share of draws:", {c: round(float(w0[(train_df.corpus == c).to_numpy()].sum()), 3) for c in sorted(train_df.corpus.unique())})
print(f"train {len(train_df)} | val core {len(VAL_CORE_FULL)} (checks use {len(VAL_CORE)}) | ElevenLabs val segments {len(EL_VAL)} "
      f"| test core {len(TEST_CORE)} | ElevenLabs test segments {len(EL_TEST)}")
_dl = make_train_loader(K, CFG["SEED"]); xb, x2b, yb, kb = next(iter(_dl)); del _dl
print("batch:", tuple(xb.shape), "labels", yb.tolist()[:8], "kinds", kb.tolist()[:8])

## 11. Model (from V3), frozen V3 teacher, and V3 weight anchors

In [ ]:
from transformers import Wav2Vec2Config, Wav2Vec2Model

class AttnStatsPool(nn.Module):
    def __init__(self, d):
        super().__init__(); self.att = nn.Sequential(nn.Linear(d, 128), nn.Tanh(), nn.Linear(128, 1))
    def forward(self, h):
        a = torch.softmax(self.att(h).float(), dim=1).to(h.dtype); mu = (a * h).sum(1)
        sd = torch.sqrt(((a * (h - mu.unsqueeze(1)) ** 2).sum(1)).clamp(min=1e-6)); return torch.cat([mu, sd], dim=-1)

class DeepfakeDetector(nn.Module):                      # identical to V2/V3 (state dicts are interchangeable)
    def __init__(self, keep_layers):
        super().__init__()
        cfg = (Wav2Vec2Config.from_json_file(CFG["ARCH_CONFIG_JSON"]) if CFG["ARCH_CONFIG_JSON"]
               else Wav2Vec2Config.from_pretrained(CFG["BASE_MODEL"]))
        cfg.num_hidden_layers = keep_layers; cfg.layerdrop = 0.0; cfg.mask_time_prob = 0.05; cfg.mask_feature_prob = 0.0
        self.backbone = Wav2Vec2Model(cfg)
        for p in self.backbone.feature_extractor.parameters(): p.requires_grad = False
        d = cfg.hidden_size
        self.layer_w = nn.Parameter(torch.zeros(keep_layers + 1))
        self.proj = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, 256), nn.GELU()); self.pool = AttnStatsPool(256)
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(512, 128), nn.GELU(), nn.Dropout(0.2), nn.Linear(128, 1))
    def forward(self, wav):
        hs = self.backbone(wav, output_hidden_states=True).hidden_states; w = torch.softmax(self.layer_w, 0)
        return self.head(self.pool(self.proj(sum(w[i] * hs[i] for i in range(len(hs)))))).squeeze(-1)

def load_state(path):
    st = torch.load(path, map_location="cpu", weights_only=False)
    return st["model"] if isinstance(st, dict) and "model" in st else st

if DEVICE == "cpu" and not CFG["SMOKE_TEST"]:
    raise SystemExit("Stop: switch to a GPU runtime before training (or set SMOKE_TEST=True for a CPU check).")
init_path = on_drive(CFG["INIT_CHECKPOINT"])
if not init_path.exists(): raise SystemExit(f"INIT_CHECKPOINT not found: {init_path}. Copy V3's export_posttrain_v3 to Drive or set the path.")
INIT_SHA = sha256(init_path)
print("init checkpoint", init_path, "sha256", INIT_SHA)
if CFG["EXPECTED_INIT_SHA256"] and INIT_SHA != CFG["EXPECTED_INIT_SHA256"]:
    print("⚠️ Not byte-identical to the V3 detector_state.pt the comparison tested. Fine only if you know these are V3's weights.")
model = DeepfakeDetector(CFG["KEEP_LAYERS"]); model.load_state_dict(load_state(init_path), strict=True); model.to(DEVICE)
INIT_STATE = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

teacher = copy.deepcopy(model).eval()                 # frozen V3: the "never worse than V3" reference
for p in teacher.parameters(): p.requires_grad_(False)
if CFG.get("_TEST_TRAIN_HEAD_ONLY"):                  # self-test hook only (CPU RAM); never set for real training
    for n, p in model.named_parameters():
        if n.startswith("backbone."): p.requires_grad_(False)
TRAINABLE = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
ANCHORS = [p.detach().clone() for _, p in TRAINABLE]  # V3 weights for the per-step anchor pull
if DEVICE == "cuda" and gpu_mem < 35:
    model.backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
print(f"params {sum(p.numel() for p in model.parameters())/1e6:.1f}M, trainable {sum(p.numel() for _, p in TRAINABLE)/1e6:.1f}M")

## 12. Evaluation suite, call-verdict replay, and the V3 baseline
`suite()` is what the Guardian sees at every check. Scores are raw float64 logits. Thresholds are always derived from real-speech validation windows pooled over conditions, the way the live system uses one threshold for everything.

`CallEvidence` mirrors the app's `CallEvidenceEngine` (`risk.py`, call-evidence-v1), so call-level results match what a senior would see.

In [ ]:
from collections import deque

def autocast():
    return torch.autocast("cuda", dtype=AMP_DTYPE or torch.float16, enabled=AMP_DTYPE is not None)

@torch.no_grad()
def predict_logits(net, dl):
    was = net.training; net.eval(); Z, Y = [], []
    for x, y in dl:
        with autocast(): Z.append(net(x.to(DEVICE, non_blocking=True)).float().cpu().double()); Y.append(y)
    net.train(was)
    return torch.cat(Y).numpy(), torch.cat(Z).numpy()

def eer_and_threshold(y, z):
    if len(np.unique(y)) < 2: return float("nan"), float("nan")
    fpr, tpr, thr = roc_curve(y, z); fnr = 1 - tpr; i = np.nanargmin(np.abs(fnr - fpr))
    return float((fpr[i] + fnr[i]) / 2), float(thr[i])

def threshold_at_fpr(y, z, target):
    real = np.sort(z[y == 0])[::-1]; k = int(np.floor(target * len(real)))
    return float(np.nextafter(real[k], np.inf)) if k < len(real) else float(real.min())

def score_frame(net, frame, conds, seed=1, expand=False):
    f = (expand_windows(frame) if expand else frame).reset_index(drop=True)
    if not len(f): return pd.DataFrame(columns=["label", "corpus", "origin", "group", "system", "cond", "z"])
    out = []
    for c in conds:
        _, z = predict_logits(net, eval_loader(f, c, seed))
        d = f[["label", "corpus", "origin", "group", "system"]].copy(); d["cond"] = c; d["z"] = z; out.append(d)
    return pd.concat(out, ignore_index=True)

NS_KINDS = {"dtmf": synth_dtmf, "music": synth_music, "white_noise": lambda n, r: colored_noise(n, r, "white"), "recorded_noise": recorded_noise}
def nonspeech_probe(net, n_each=None, seed=5):
    n_each = n_each or (4 if CFG["SMOKE_TEST"] else 40); rng = np.random.default_rng(seed)
    x = np.stack([normalize(fn(WIN, rng) * .3) for fn in NS_KINDS.values() for _ in range(n_each)])
    was = net.training; net.eval(); z = []
    with torch.no_grad(), autocast():
        for i in range(0, len(x), 16): z.append(net(torch.from_numpy(x[i:i + 16]).to(DEVICE)).float().cpu().double().numpy())
    net.train(was)
    return np.concatenate(z)

def metrics_from(core, el, ns, thr=None):
    """thr=None: derive thresholds from these real windows (validation). thr=(eer, target): use a model's fixed thresholds (test)."""
    y, z = core.label.to_numpy().astype(int), core.z.to_numpy()
    pooled_eer, t_eer = eer_and_threshold(y, z)
    t_target = threshold_at_fpr(y, z, CFG["TARGET_FPR"]); t_1pct = threshold_at_fpr(y, z, 0.01)
    if thr is not None: t_eer, t_target = thr
    real, fake = core[core.label == 0], core[core.label == 1]
    ls = real[real.corpus == "LibriSpeech"]
    m = dict(pooled_eer=pooled_eer, thr_eer=t_eer, thr_target=t_target, thr_1pct=t_1pct,
             cond_eer={c: eer_and_threshold(d.label.to_numpy().astype(int), d.z.to_numpy())[0] for c, d in core.groupby("cond")},
             tpr_other=float((fake.z >= t_target).mean()) if len(fake) else float("nan"),
             real_fpr=float((real.z >= t_target).mean()), real_fpr_eer=float((real.z >= t_eer).mean()),
             ls_fpr=float((ls.z >= t_target).mean()) if len(ls) else 0.0,
             nonspeech_max=float(ns.max()), nonspeech_flag=float((ns >= t_eer).mean()))
    if len(el):
        m.update(el_det=float((el.z >= t_target).mean()), el_det_eer=float((el.z >= t_eer).mean()),
                 el_det_by_cond={c: float((d.z >= t_target).mean()) for c, d in el.groupby("cond")},
                 el_det_by_origin={o: float((d.z >= t_eer).mean()) for o, d in el.groupby("origin")},
                 el_auc=float(roc_auc_score(np.r_[np.zeros(len(real)), np.ones(len(el))], np.r_[real.z, el.z])))
    m["objective"] = m.get("el_det", m["tpr_other"])
    return m

def suite(net, quick=True, core_frame=None, el_frame=None):
    core = score_frame(net, VAL_CORE if core_frame is None else core_frame, CFG["SELECT_CONDITIONS"] if quick else EVAL_CONDITIONS)
    el = score_frame(net, EL_VAL if el_frame is None else el_frame, BENCH_CONDITIONS, expand=True)
    return metrics_from(core, el, nonspeech_probe(net)), (core, el)

class CallEvidence:
    """Mirror of risk.CallEvidenceEngine (call-evidence-v1), fed raw logits."""
    def __init__(self, low_z, high_z):
        self.center = (low_z + high_z) / 2; self.scale = max((high_z - low_z) / (2 * math.log(.65 / .35)), .25)
        self.total, self.n, self.state, self.cand, self.cand_steps = 0.0, 0, "analyzing", None, 0
        self.recent, self.warned = deque(maxlen=5), False
    def update(self, z):
        b = max(-4.0, min(4.0, (z - self.center) / self.scale)); self.total += b; self.n += 1; self.recent.append(b)
        sig = lambda v: 1 / (1 + math.exp(-v)); score = 100 * sig(self.total / self.n)
        target = "human" if score <= 35 else "ai" if score >= 65 else "uncertain"
        if self.n >= 2:
            if self.state == "analyzing": self.state = target
            elif target != self.state:
                self.cand_steps = self.cand_steps + 1 if self.cand == target else 1; self.cand = target
                if self.cand_steps >= 2: self.state, self.cand_steps = target, 0
            else: self.cand, self.cand_steps = None, 0
        rm = sum(self.recent) / len(self.recent)
        self.warned |= len(self.recent) >= 3 and 100 * sig(rm) >= 65 and sum(100 * sig(v) >= 65 for v in self.recent) >= 3
        return self.state

def recentred(t_eer, t_1pct):
    """Call-engine thresholds centred on the validated EER threshold (same spread). For V3: 0.046875 / 4.015625."""
    half = (t_1pct - t_eer) / 2
    return t_eer - half, t_eer + half

baseline_path = CKPT_DIR / "baseline_v3_suite.json"
data_sig = hashlib.sha256((signature + str(len(VAL_CORE)) + str(len(EL_VAL)) + json.dumps(CFG["SELECT_CONDITIONS"])).encode()).hexdigest()
if baseline_path.exists() and json.loads(baseline_path.read_text()).get("data_sig") == data_sig:
    BASELINE = json.loads(baseline_path.read_text())["metrics"]; print("loaded cached V3 baseline")
else:
    print("Scoring V3 (the starting point) with the Guardian's validation suite...")
    BASELINE, _ = suite(teacher)
    save_json(baseline_path, dict(data_sig=data_sig, metrics=BASELINE))
def show(m, title):
    keys = ["objective", "pooled_eer", "tpr_other", "ls_fpr", "real_fpr", "nonspeech_flag", "nonspeech_max", "el_det", "el_det_eer", "el_auc", "thr_eer", "thr_target"]
    print(title, {k: round(m[k], 4) for k in keys if k in m and m[k] == m[k]})
    print("   condition EER:", {c: round(v, 4) for c, v in m["cond_eer"].items()})
    if "el_det_by_cond" in m: print("   ElevenLabs detected by condition @target:", {c: round(v, 3) for c, v in m["el_det_by_cond"].items()})
show(BASELINE, "V3 baseline (validation):")

## 13. The Guardian: self-correction rules
At every check, each guarded metric is compared with **two references**: V3 (the floor) and the **best accepted state** (the ratchet). Any metric that gets worse by more than its tolerance counts as a violation.

| Guard | Direction | Tolerance |
|---|---|---|
| Pooled validation EER | lower | `TOL_POOLED_EER` |
| Each condition's EER | lower | `TOL_COND_EER` |
| Other-TTS detection @ 2% false flags | higher | `TOL_TPR` |
| LibriSpeech (unseen speakers) false flags @ 2% threshold | lower | `TOL_LS_FPR` |
| Non-speech probes flagged | lower | `TOL_NONSPEECH` (2 of 160 probes) |
| ElevenLabs objective | higher | `TOL_OBJECTIVE` |

What the Guardian does on a violation:
1. Rolls back to the best weights.
2. Resets the optimizer.
3. Multiplies the learning rate by `LR_BACKOFF`.
4. Retunes the knobs tied to whatever broke.
5. Rebuilds the loader.

After `MAX_CORRECTIONS` rollbacks, it stops and keeps the best state.

In [ ]:
def clamp_knobs(K):
    for k, (lo, hi) in CFG["KNOB_LIMITS"].items():
        if k in K: K[k] = float(min(hi, max(lo, K[k])))
    return K

# metric -> (direction, tolerance); direction +1 = higher is better
def guard_specs(m):
    specs = {"pooled_eer": (-1, CFG["TOL_POOLED_EER"]), "tpr_other": (+1, CFG["TOL_TPR"]),
             "ls_fpr": (-1, CFG["TOL_LS_FPR"]), "nonspeech_flag": (-1, CFG["TOL_NONSPEECH"]), "objective": (+1, CFG["TOL_OBJECTIVE"])}
    specs.update({f"cond:{c}": (-1, CFG["TOL_COND_EER"]) for c in m["cond_eer"]})
    return specs

def metric_value(m, name):
    return m["cond_eer"][name[5:]] if name.startswith("cond:") else m[name]

def fixes_for(name, K):
    """What to change when a guard fails. Returns a short description."""
    if name in ("cond:noise10", "cond:noise20", "cond:phone_noise10"):
        K["NOISE_P"] += 0.05; K["CONSIST_WEIGHT"] *= 1.5; return "more noise augmentation + stronger noise consistency"
    if name == "cond:gap2s": K["GAP_P"] += 0.05; return "more silence-gap augmentation"
    if name == "cond:music10": K["OVERLAY_P"] += 0.05; return "more background overlays"
    if name == "nonspeech_flag": K["NONSPEECH_AS_REAL_P"] += 0.02; return "more non-speech labelled not-AI"
    if name == "objective": K["ELEVEN_FAKE_SHARE"] *= 1.25; return "more ElevenLabs in each batch"
    # clean/phone/pooled/other-TTS/LibriSpeech: the new data is pulling the model away from what V3 did well
    K["KD_WEIGHT"] *= 1.5; K["ANCHOR_RATE"] = max(K["ANCHOR_RATE"] * 1.5, 1e-4)
    if name != "ls_fpr": K["ELEVEN_FAKE_SHARE"] *= 0.8
    return "stronger V3 distillation + anchor" + ("" if name == "ls_fpr" else ", less ElevenLabs pressure")

class Guardian:
    def __init__(self, baseline):
        self.base, self.ref, self.best_obj = baseline, baseline, baseline["objective"]
        self.corrections = self.boosts = self.stall = self.checks = 0
        self.best_check, self.log = 0, []
    def violations(self, m):
        out = {}
        for name, (d, tol) in guard_specs(self.base).items():
            v = metric_value(m, name)
            if v != v: continue                                            # NaN (e.g. a condition with one class)
            for ref_name, ref in (("V3", self.base), ("best", self.ref)):
                r = metric_value(ref, name)
                if r == r and (v < r - tol if d > 0 else v > r + tol):
                    out[name] = dict(value=round(v, 4), ref=ref_name, limit=round(r - tol if d > 0 else r + tol, 4))
        if CFG.get("_TEST_FORCE_VIOLATION_AT") == self.checks:            # self-test hook only
            out["cond:noise10"] = dict(value=None, ref="forced-by-self-test", limit=None)
        return out
    def decide(self, m, step, K):
        self.checks += 1; v = self.violations(m); entry = dict(check=self.checks, step=step, objective=round(m["objective"], 4))
        if not v and m["objective"] >= self.best_obj + CFG["MIN_OBJECTIVE_GAIN"]:
            self.best_obj, self.ref, self.best_check, self.stall = m["objective"], m, self.checks, 0
            entry.update(action="accept", note="all guards pass, ElevenLabs objective improved")
        elif not v:
            self.stall += 1; entry.update(action="continue", note=f"guards pass, no gain ({self.stall}/{CFG['STALL_CHECKS']})")
            if self.stall >= CFG["STALL_CHECKS"] and self.boosts < CFG["MAX_BOOSTS"]:
                self.boosts += 1; self.stall = 0; K["ELEVEN_FAKE_SHARE"] *= 1.25; clamp_knobs(K)
                entry.update(action="boost", note="stalled: more ElevenLabs in each batch")
        else:
            self.corrections += 1; K["LR_SCALE"] *= CFG["LR_BACKOFF"]
            notes = sorted({fixes_for(name, K) for name in v}); clamp_knobs(K)
            stop = self.corrections > CFG["MAX_CORRECTIONS"]
            entry.update(action="stop" if stop else "rollback", violations=v, fixes=notes)
        entry["knobs"] = {k: round(val, 5) for k, val in K.items()}
        self.log.append(entry)
        print(f"  Guardian check {self.checks} @ step {step}: {entry['action'].upper()} | objective {m['objective']:.4f} "
              f"(best {self.best_obj:.4f}) | {entry.get('note', '')}{' violations: ' + ', '.join(v) if v else ''}")
        return entry["action"]
    def state(self): return {k: getattr(self, k) for k in ("base", "ref", "best_obj", "corrections", "boosts", "stall", "checks", "best_check", "log")}
    def load(self, s):
        for k, val in s.items(): setattr(self, k, val)
        return self
print("Guardian guards:", list(guard_specs(BASELINE)))

## 14. Fine-tune with Guardian checks (auto-resumes from Drive)
Loss = BCE + `CONSIST_WEIGHT` × noisy-twin consistency + `KD_WEIGHT` × one-sided V3 distillation. After each optimizer step, an anchor pull back toward V3 is applied.

`best.pt` starts as V3 itself (check 0). It only changes when the Guardian accepts a better state.

In [ ]:
def new_optimizer():
    head = [p for n, p in TRAINABLE if not n.startswith("backbone.")]; bb = [p for n, p in TRAINABLE if n.startswith("backbone.")]
    return torch.optim.AdamW([{"params": bb, "lr": CFG["LR_BACKBONE"], "base_lr": CFG["LR_BACKBONE"]},
                              {"params": head, "lr": CFG["LR_HEAD"], "base_lr": CFG["LR_HEAD"]}], weight_decay=CFG["WEIGHT_DECAY"])
def new_scaler(): return torch.amp.GradScaler("cuda", enabled=(AMP_DTYPE == torch.float16))

steps_per_epoch = max(1, len(train_df) // BATCH // ACCUM)
if CFG["SMOKE_TEST"]: steps_per_epoch = min(steps_per_epoch, int(CFG.get("SMOKE_STEPS_PER_EPOCH", 4)))
total_steps = steps_per_epoch * CFG["MAX_EPOCHS"]; warm = max(1, int(total_steps * CFG["WARMUP_FRAC"]))
CHECK_EVERY = max(1, steps_per_epoch // CFG["CHECKS_PER_EPOCH"])
def lr_factor(step): return min(1.0, (step + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, step / total_steps)))
print(f"{steps_per_epoch} optimizer steps/epoch, {total_steps} max, Guardian check every {CHECK_EVERY} steps")

last_ckpt, best_ckpt, log_path = CKPT_DIR / "last.pt", CKPT_DIR / "best.pt", CKPT_DIR / "self_correction_log.json"
guardian = Guardian(BASELINE); opt, scaler = new_optimizer(), new_scaler(); opt_step, history = 0, []
if last_ckpt.exists():
    ck = torch.load(last_ckpt, map_location="cpu", weights_only=False)
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
    opt_step, history = ck["opt_step"], ck["history"]; K.update(ck["knobs"]); guardian.load(ck["guardian"])
    print(f"Resumed at step {opt_step} | knobs {K} | corrections {guardian.corrections}")
if not best_ckpt.exists():
    torch.save({"model": INIT_STATE, "metrics": BASELINE, "check": 0, "step": 0, "init_sha256": INIT_SHA}, best_ckpt)
BEST_STATE = torch.load(best_ckpt, map_location="cpu", weights_only=False)["model"]

def save_last():
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(), "opt_step": opt_step,
                "history": history, "knobs": dict(K), "guardian": guardian.state()}, last_ckpt)
    save_json(log_path, guardian.log)

def zero(z): return z.sum() * 0.0
stopped = guardian.log[-1]["action"] == "stop" if guardian.log else False
train_dl = make_train_loader(K, CFG["SEED"] + guardian.checks)
assert len(train_dl) > 0, "training set smaller than one batch"
t0 = time.time(); run = None
while opt_step < total_steps and not stopped:
    model.train(); opt.zero_grad(set_to_none=True); rebuild = False
    pbar = tqdm(train_dl, desc=f"step {opt_step}/{total_steps}", leave=False)
    for micro, (x, x2, y, kind) in enumerate(pbar):
        x, x2, y, kind = (t.to(DEVICE, non_blocking=True) for t in (x, x2, y, kind))
        B = x.size(0); k = int(round(B * CFG["CONSIST_FRAC"]))
        twin = torch.nonzero(kind[:k] != 3).flatten()                    # speech windows that also get a noisy twin
        with autocast():
            z_all = model(torch.cat([x, x2[twin]])).float()
            with torch.no_grad(): zt = teacher(x).float()
        z, z2 = z_all[:B], z_all[B:]
        l_bce = F.binary_cross_entropy_with_logits(z, y)
        l_cons = F.smooth_l1_loss(z2, z[twin].detach()) if len(twin) else zero(z)
        real, mlaad = kind == 2, kind == 0                                # one-sided: only penalise doing worse than V3
        l_kd = ((F.smooth_l1_loss(F.relu(z[real] - zt[real]), torch.zeros_like(z[real])) if real.any() else zero(z)) +
                (F.smooth_l1_loss(F.relu(zt[mlaad] - z[mlaad]), torch.zeros_like(z[mlaad])) if mlaad.any() else zero(z)))
        loss = l_bce + K["CONSIST_WEIGHT"] * l_cons + K["KD_WEIGHT"] * l_kd
        scaler.scale(loss / ACCUM).backward()
        parts = torch.stack([loss.detach(), l_bce.detach(), l_cons.detach(), l_kd.detach()]).cpu().numpy()
        run = parts if run is None else 0.98 * run + 0.02 * parts
        if (micro + 1) % ACCUM: continue
        f = lr_factor(opt_step)
        for g in opt.param_groups: g["lr"] = g["base_lr"] * K["LR_SCALE"] * f
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_([p for _, p in TRAINABLE], CFG["GRAD_CLIP"])
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); opt_step += 1
        if K["ANCHOR_RATE"] > 0:                                          # pull weights slightly back toward V3
            with torch.no_grad():
                for (_, p), a in zip(TRAINABLE, ANCHORS): p.lerp_(a, K["ANCHOR_RATE"] * f)
        pbar.set_postfix(loss=f"{run[0]:.3f}", bce=f"{run[1]:.3f}", cons=f"{run[2]:.3f}", kd=f"{run[3]:.3f}", lr_scale=K["LR_SCALE"])
        if opt_step % CHECK_EVERY == 0 or opt_step >= total_steps:
            m, _ = suite(model)
            action = guardian.decide(m, opt_step, K)
            history.append(dict(step=opt_step, epoch=round(opt_step / steps_per_epoch, 2), action=action, loss=float(run[0]),
                                objective=m["objective"], pooled_eer=m["pooled_eer"], tpr_other=m["tpr_other"], ls_fpr=m["ls_fpr"],
                                nonspeech_flag=m["nonspeech_flag"], **{f"eer_{c}": v for c, v in m["cond_eer"].items()},
                                minutes=(time.time() - t0) / 60, **{f"k_{a}": b for a, b in K.items()}))
            if action == "accept":
                BEST_STATE = {kk: vv.detach().cpu().clone() for kk, vv in model.state_dict().items()}
                torch.save({"model": BEST_STATE, "metrics": m, "check": guardian.checks, "step": opt_step, "init_sha256": INIT_SHA}, best_ckpt)
                print("  ✓ new best saved to Drive"); show(m, "  accepted:")
            elif action in ("rollback", "stop"):
                model.load_state_dict(BEST_STATE); opt, scaler = new_optimizer(), new_scaler(); rebuild = True
                print(f"  ↺ rolled back to best (check {guardian.best_check}); fixes: {guardian.log[-1]['fixes']}")
                stopped = action == "stop"
            elif action == "boost":
                rebuild = True
            save_last()
            if rebuild or stopped or opt_step >= total_steps: break
            model.train()
    pbar.close()
    if rebuild and not stopped:
        train_dl = make_train_loader(K, CFG["SEED"] + guardian.checks)      # new knobs reach the workers
print(f"training finished at step {opt_step}: {guardian.corrections} corrections, {guardian.boosts} boosts, best = check {guardian.best_check}")
if history: print(pd.DataFrame(history)[["step", "epoch", "action", "objective", "pooled_eer", "tpr_other", "ls_fpr", "nonspeech_flag"]].round(4).to_string(index=False))

## 15. Final evaluation: thresholds, test set, and the release gate
- **Thresholds** come from the full **validation** set, pooled over every condition.
- **Test** data (unseen TTS systems, unseen LibriSpeech speakers, held-out ElevenLabs voices) is scored here for the first time.
- For the gate, V3 and V4 each get thresholds derived the same way from the same validation set, so it compares the models rather than threshold choices. Section 16 shows V3 with its deployed config thresholds.

The **release gate** checks that V4 is no worse than V3 on every test metric. If any check fails, the export is marked do-not-deploy.

In [ ]:
import matplotlib.pyplot as plt
best = torch.load(best_ckpt, map_location="cpu", weights_only=False)
model.load_state_dict(best["model"]); model.eval()
KEPT_CHECK = best["check"]
print("kept state: Guardian check", KEPT_CHECK, "(0 = no checkpoint beat V3 on every guard; V4 == V3)")

v4_val, _ = suite(model, quick=False, core_frame=VAL_CORE_FULL)
v3_val, _ = suite(teacher, quick=False, core_frame=VAL_CORE_FULL)
LOGIT_EER_T, LOGIT_1PCT_T, LOGIT_TARGET_T = v4_val["thr_eer"], v4_val["thr_1pct"], v4_val["thr_target"]
# Gate compares MODELS, so both get thresholds derived the same way on the same validation set
# (V3's config thresholds came from V3's older validation set; the benchmark table in section 16 uses those).
V3_T = (v3_val["thr_eer"], v3_val["thr_target"])
print(f"V4 logit thresholds (validation, pooled): EER {LOGIT_EER_T:.3f} | 2% FPR {LOGIT_TARGET_T:.3f} | 1% FPR {LOGIT_1PCT_T:.3f}")
for t in (LOGIT_EER_T, LOGIT_1PCT_T):
    if t >= 27.6: print("⚠️ threshold above the live engine's logit clamp (27.6). Report this before deploying.")
show(v3_val, "V3 validation (full):"); show(v4_val, "V4 validation (full):")

def test_eval(net, thr):
    core = score_frame(net, TEST_CORE, EVAL_CONDITIONS, seed=7)
    el = score_frame(net, EL_TEST, BENCH_CONDITIONS, seed=7, expand=True)
    m = metrics_from(core, el, nonspeech_probe(net, seed=11), thr=thr)
    m["per_system"] = core[core.label == 1].groupby("system").z.apply(lambda s: float((s >= thr[1]).mean())).to_dict()
    return m, core, el
v3_test, v3_core_t, v3_el_t = test_eval(teacher, V3_T)
v4_test, v4_core_t, v4_el_t = test_eval(model, (LOGIT_EER_T, LOGIT_TARGET_T))
show(v3_test, "V3 TEST:"); show(v4_test, "V4 TEST:")

gate = []
def check(name, v3, v4, better, tol):
    if v3 != v3 or v4 != v4: gate.append(dict(metric=name, V3=v3, V4=v4, result="n/a")); return
    ok = v4 <= v3 + tol if better == "lower" else v4 >= v3 - tol
    gate.append(dict(metric=name, V3=round(v3, 4), V4=round(v4, 4), result="PASS" if ok else "FAIL"))
check("pooled test EER", v3_test["pooled_eer"], v4_test["pooled_eer"], "lower", CFG["TOL_POOLED_EER"])
for c in EVAL_CONDITIONS: check(f"test EER: {c}", v3_test["cond_eer"].get(c, np.nan), v4_test["cond_eer"].get(c, np.nan), "lower", CFG["TOL_COND_EER"])
check("unseen-TTS detected @ own 2% threshold", v3_test["tpr_other"], v4_test["tpr_other"], "higher", CFG["TOL_TPR"])
check("real speech flagged @ own EER threshold", v3_test["real_fpr_eer"], v4_test["real_fpr_eer"], "lower", 0.01)
check("LibriSpeech (unseen speakers) flagged @ own 2% thr", v3_test["ls_fpr"], v4_test["ls_fpr"], "lower", CFG["TOL_LS_FPR"])
check("non-speech probes flagged", v3_test["nonspeech_flag"], v4_test["nonspeech_flag"], "lower", CFG["TOL_NONSPEECH"])
if "el_det_eer" in v4_test:
    check("ElevenLabs held-out detected @ own EER thr", v3_test["el_det_eer"], v4_test["el_det_eer"], "higher", 0.0)
    for o in v4_test["el_det_by_origin"]:
        print(f"  ElevenLabs held-out [{o}] detected @ own EER thr: V3 {v3_test['el_det_by_origin'][o]:.3f} -> V4 {v4_test['el_det_by_origin'][o]:.3f}")
GATE = pd.DataFrame(gate); RELEASE_OK = bool(KEPT_CHECK > 0 and (GATE.result != "FAIL").all())
print("\nRELEASE GATE (test set, V4 must not be worse than V3):\n", GATE.to_string(index=False))
print("\n" + ("✅ PASS: V4 is no worse than V3 on every test metric." if RELEASE_OK else
              "⛔ NOT RELEASED: " + ("no checkpoint beat V3." if KEPT_CHECK == 0 else "at least one test metric regressed.") +
              " Keep V3; the export is marked CANDIDATE_DO_NOT_DEPLOY."))

EXPORT = DRIVE / (f"export_{CFG['RUN_TAG']}{sfx}" + ("" if RELEASE_OK else "_CANDIDATE_DO_NOT_DEPLOY")); EXPORT.mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
cond = pd.DataFrame({"V3": v3_test["cond_eer"], "V4": v4_test["cond_eer"]}) * 100
cond.plot.bar(ax=ax[0]); ax[0].set_ylabel("test EER (%)"); ax[0].set_title("Robustness, test set")
ax[1].hist(v4_core_t[v4_core_t.label == 0].z, 60, alpha=.5, label="real (test)"); ax[1].hist(v4_core_t[v4_core_t.label == 1].z, 60, alpha=.5, label="other TTS (test)")
if len(v4_el_t): ax[1].hist(v4_el_t.z, 30, alpha=.6, label="ElevenLabs held-out")
ax[1].axvline(LOGIT_EER_T, c="k", ls="--", label="V4 EER thr"); ax[1].set_xlabel("V4 logit"); ax[1].legend()
if history:
    h = pd.DataFrame(history); ax[2].plot(h.step, h.objective, "o-", label="ElevenLabs objective (val)")
    ax[2].axhline(BASELINE["objective"], c="gray", ls=":", label="V3")
    for _, r in h[h.action.isin(["rollback", "stop"])].iterrows(): ax[2].axvline(r.step, c="r", alpha=.3)
    ax[2].set_xlabel("step (red = Guardian rollback)"); ax[2].legend()
plt.tight_layout(); plt.savefig(EXPORT / "v4_results.png", dpi=120); plt.show()

## 16. The repo benchmark, rebuilt: V2 vs V3 vs V4 (README-style table)
Recreates `model teseting/Compare V2 and V3` on the files in `benchmark_audio/`: the same conditions, seed, PCM16 step, 4 s windows, and each model's **own** EER threshold.

It also replays every case through the app's call verdict, with each model's thresholds and with the re-centred pair. Voices that were used in training are reported separately, because only unseen voices are a fair test.

In [ ]:
def bench_decode(path):
    try: x, rate = sf.read(str(path), dtype="float32", always_2d=True)
    except Exception: return load_audio(path)                 # older libsndfile without MP3: ffmpeg decode
    x = x.mean(1)
    if rate != SR: g = math.gcd(rate, SR); x = resample_poly(x, SR // g, rate // g).astype(np.float32)
    return x

def bench_conditions(x):                       # = model teseting/Compare V2 and V3/sources/evaluate.py conditions()
    rng = np.random.default_rng(20261022)
    out = {"original": x, "8khz_bandwidth_only": band8k(x), "g711_mulaw": g711(x), "low_volume_minus20dB": x * .1,
           "mild_clipping": np.clip(x / np.quantile(np.abs(x), .99), -1, 1)}
    for snr in [30, 20, 10]:
        n = rng.standard_normal(len(x)).astype(np.float32); n *= np.sqrt(np.mean(x * x) / (10 ** (snr / 10) * np.mean(n * n)))
        out[f"white_noise_{snr}dB_SNR"] = x + n
    for gap in [2, 10]: out[f"pause_{gap}s_after_16s"] = np.concatenate([x[:16 * SR], np.zeros(gap * SR, np.float32), x[16 * SR:]])
    return out

BENCH_ROWS, BENCH_X = [], []
for p in audio_files(BENCH_DIR):
    is_ai = "elevenlabs" in p.name.lower(); x = bench_decode(p)
    if not is_ai: x = x[:40 * SR]
    voice = voice_of(p, BENCH_DIR) if is_ai else p.stem
    seen = sha256(p) in TRAINED_FILE_SHA or (is_ai and voice in TRAINED_VOICES)
    for cond, s in bench_conditions(x).items():
        s = (np.clip(s, -1, 1) * 32767).astype(np.int16).astype(np.float32) / 32768
        gap = int(cond.split("_")[1][:-1]) if cond.startswith("pause_") else 0
        for st in range(0, len(s) - WIN + 1, WIN):
            a, b = st / SR, (st + WIN) / SR
            if gap and a < 16 + gap and b > 16: continue              # windows touching the inserted pause are not speech-only
            BENCH_ROWS.append(dict(file=p.name, voice=voice, label=int(is_ai), cond=cond, start=a, seen_in_training=seen))
            BENCH_X.append(normalize(s[st:st + WIN]))
if CFG["SMOKE_TEST"] and BENCH_ROWS:                                   # keep the CPU self-test short
    keep = [i for i, r in enumerate(BENCH_ROWS) if r["cond"] in ("original", "white_noise_10dB_SNR") and r["start"] < 12]
    BENCH_ROWS, BENCH_X = [BENCH_ROWS[i] for i in keep], [BENCH_X[i] for i in keep]

@torch.no_grad()
def score_array(net, X):
    net.eval(); out = []
    for i in range(0, len(X), BATCH * 2):
        with autocast(): out.append(net(torch.from_numpy(np.stack(X[i:i + BATCH * 2])).to(DEVICE)).float().cpu().double().numpy())
    return np.concatenate(out)

def bench_table(B, models):
    rows = []
    for name, (t_eer, t_1pct) in models.items():
        for subset, d in (("all speech conditions", B), ("original only", B[B.cond == "original"]),
                          ("white noise only", B[B.cond.str.startswith("white")]), ("unseen voices only", B[~B.seen_in_training])):
            if not len(d) or d.label.nunique() < 2: continue
            y, z = d.label.to_numpy(), d[name].to_numpy(); pred = z >= t_eer
            ai, fp = pred[y == 1].mean(), pred[y == 0].mean()
            rows.append(dict(subset=subset, model=name, windows=len(d), accuracy=(pred == y).mean(), balanced=(ai + 1 - fp) / 2,
                             human_flagged=fp, elevenlabs_detected=ai, auc=roc_auc_score(y, z), eer=eer_and_threshold(y, z)[0]))
    return pd.DataFrame(rows)

def call_replay(B, name, low, high):
    c = {}
    for (f, cond), d in B.sort_values("start").groupby(["file", "cond"]):
        e = CallEvidence(low, high)
        for z in d[name]: e.update(float(z))
        key = ("ElevenLabs" if d.label.iloc[0] else "human") + " -> " + e.state
        c[key] = c.get(key, 0) + 1
        if e.warned: c[("ElevenLabs" if d.label.iloc[0] else "human") + " warned"] = c.get(("ElevenLabs" if d.label.iloc[0] else "human") + " warned", 0) + 1
    return dict(sorted(c.items()))

BENCH = None
if BENCH_ROWS:
    BENCH = pd.DataFrame(BENCH_ROWS); MODELS = {}
    if on_drive(CFG["V2_CHECKPOINT"]).exists():
        v2 = DeepfakeDetector(CFG["KEEP_LAYERS"]); v2.load_state_dict(load_state(on_drive(CFG["V2_CHECKPOINT"]))); v2.to(DEVICE)
        BENCH["V2"] = score_array(v2, BENCH_X); MODELS["V2"] = tuple(CFG["V2_LOGIT_THRESHOLDS"]); del v2
    BENCH["V3"] = score_array(teacher, BENCH_X); MODELS["V3"] = tuple(CFG["V3_LOGIT_THRESHOLDS"])
    BENCH["V3 (same-val thr)"] = BENCH["V3"]; MODELS["V3 (same-val thr)"] = (v3_val["thr_eer"], v3_val["thr_1pct"])
    BENCH["V4"] = score_array(model, BENCH_X); MODELS["V4"] = (LOGIT_EER_T, LOGIT_1PCT_T)
    BENCH_TABLE = bench_table(BENCH, MODELS)
    pct = lambda v: f"{v*100:.1f}%"
    print("Windows at each model's own EER threshold (cf. README):")
    print(BENCH_TABLE.assign(**{c: BENCH_TABLE[c].map(pct) for c in ["accuracy", "balanced", "human_flagged", "elevenlabs_detected", "eer"]},
                             auc=BENCH_TABLE.auc.round(4)).to_string(index=False))
    print("'V3' uses V3's deployed config thresholds (as in the README). 'V3 (same-val thr)' re-derives V3's thresholds on the same\n"
          "validation set as V4, so V4 minus that row is the gain from the new weights alone.")
    seen = BENCH[BENCH.seen_in_training].voice.unique()
    if len(seen): print(f"⚠️ voices seen in training (not a fair test): {sorted(seen)}. Use the 'unseen voices only' rows.")
    CALLS = {}
    for name, (t_eer, t_1pct) in MODELS.items():
        CALLS[f"{name} (its thresholds)"] = call_replay(BENCH, name, t_eer, t_1pct)
        CALLS[f"{name} (re-centred)"] = call_replay(BENCH, name, *recentred(t_eer, t_1pct))
    print("\nCall-verdict replay (each benchmark case through the app's CallEvidenceEngine logic):")
    for k, v in CALLS.items(): print(f"  {k:28s} {v}")
    BENCH.to_csv(EXPORT / "benchmark_windows.csv", index=False); BENCH_TABLE.to_csv(EXPORT / "benchmark_table.csv", index=False)
else:
    print(f"No files in {BENCH_DIR}; copy the 5 files from 'model teseting/Compare V2 and V3/inputs/audio/' to get the README-style table.")

OWN_EVAL = DRIVE / CFG["OWN_EVAL_SUBDIR"]                              # optional eval-only recordings
if OWN_EVAL.exists():
    rows = []
    for lab in ("real", "fake"):
        for src in sorted(p for p in (OWN_EVAL / lab).glob("*") if p.is_dir()):
            W = [normalize(x[i:i + WIN]) for f in audio_files(src) for x in [load_audio(f)] for i in range(0, len(x) - WIN + 1, WIN)]
            if not W: continue
            z3, z4 = score_array(teacher, W), score_array(model, W)
            rows.append(dict(label=lab, source=src.name, windows=len(W), V3_flagged=float((z3 >= CFG["V3_LOGIT_THRESHOLDS"][0]).mean()),
                             V4_flagged=float((z4 >= LOGIT_EER_T).mean())))
    if rows: print("\nEval-only recordings (share of windows flagged AI at each model's EER threshold):\n", pd.DataFrame(rows).round(3).to_string(index=False))

## 17. Latency check (same architecture as V2/V3)

In [ ]:
model.eval(); xl = torch.randn(1, WIN, device=DEVICE)
def bench_ms(fn, n=30):
    for _ in range(3): fn()
    if DEVICE == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(n): fn()
    if DEVICE == "cuda": torch.cuda.synchronize()
    return (time.time() - t0) / n * 1000
with torch.no_grad():
    if DEVICE == "cuda":
        with autocast(): print(f"GPU ({gpu_name}): {bench_ms(lambda: model(xl)):.1f} ms per 4 s window")
    print("Same architecture and size as V3, so CPU/MPS latency in the app is unchanged.")

## 18. Export a bundle the repo can load
Writes the following to `export_posttrain_v4/`, or to `export_posttrain_v4_CANDIDATE_DO_NOT_DEPLOY/` if the release gate failed:
- `detector_state.pt`
- `detector_config.json` (same keys the app reads, plus the V4 details and the recommended call-engine thresholds)
- `posttrain_report.json`
- `self_correction_log.json`
- the benchmark CSVs
- `v4_results.png`
- `SHA256SUMS.txt`

Don't edit `detector_config.json` after download: the agent verifies its SHA-256.

In [ ]:
def sigmoid64(z): return float(1 / (1 + math.exp(-z))) if z >= 0 else float(math.exp(z) / (1 + math.exp(z)))
p_eer, p_1pct = sigmoid64(LOGIT_EER_T), sigmoid64(LOGIT_1PCT_T)
assert 0 < p_eer <= p_1pct < 1, "probability thresholds must be strictly below 1 (float64)"
REC_LOW, REC_HIGH = recentred(LOGIT_EER_T, LOGIT_1PCT_T)
data_summary = (comm.groupby(["corpus", "origin", "split"]).agg(segments=("seg", "size"), voices=("group", "nunique")).reset_index().to_dict("records")
                if len(comm) else [])
meta = dict(
    base_model=CFG["BASE_MODEL"], keep_layers=CFG["KEEP_LAYERS"], sample_rate=SR, window_sec=CFG["TRAIN_SEC"],
    threshold_eer=p_eer, threshold_1pct_fpr=p_1pct,
    logit_threshold_eer=LOGIT_EER_T, logit_threshold_1pct_fpr=LOGIT_1PCT_T, logit_threshold_2pct_fpr=LOGIT_TARGET_T,
    threshold_selection="validation windows pooled over all robustness conditions, raw float64 logits",
    val_eer=v4_val["pooled_eer"], val_eer_by_condition=v4_val["cond_eer"],
    test=dict(pooled=dict(eer=v4_test["pooled_eer"]), **{c: dict(eer=e) for c, e in v4_test["cond_eer"].items()}),
    unseen_test_systems=sorted(test_df.query("label==1 and corpus=='MLAAD'").system.unique()),
    score_meaning="sigmoid(logit) = probability the audio is AI-generated (uncalibrated)",
    trained=time.strftime("%Y-%m-%d"),
    recommended_call_engine_env=dict(THRESHOLD_ELEVATED_LOGIT=REC_LOW, THRESHOLD_HIGH_LOGIT=REC_HIGH,
        why="Centres the app's call verdict on the validated EER threshold with the same spread (see Compare V2 and V3/experiments)."),
    posttrain=dict(version="v4", init_checkpoint_sha256=INIT_SHA, kept_guardian_check=KEPT_CHECK, release_gate_passed=RELEASE_OK,
                   data_mode=MODE, commercial_data=data_summary, benchmark_voices_held_out=(MODE == "external"),
                   trained_voices=TRAINED_VOICES, final_knobs=dict(K), corrections=guardian.corrections, boosts=guardian.boosts,
                   real_corpora=sorted(manifest.query("label==0").corpus.unique()), excluded_speakers=sorted(BENCHMARK_SPEAKERS),
                   losses="BCE + noisy-twin consistency + one-sided V3 distillation; per-step anchor to V3 weights",
                   caveat=("Guardian selection used validation only; test and benchmark were scored once after training. "
                           "LibriSpeech controls are in-corpus; external ElevenLabs language not verified; use own recordings and live calls "
                           "for an out-of-corpus check.")),
)
torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, EXPORT / "detector_state.pt")
(EXPORT / "detector_config.json").write_text(json.dumps(meta, indent=2))
report = dict(v3_baseline_val_quick=BASELINE, val_full=dict(V3=v3_val, V4=v4_val), test=dict(V3=v3_test, V4=v4_test),
              release_gate=gate, guardian_history=history, benchmark_table=None if BENCH is None else BENCH_TABLE.to_dict("records"),
              benchmark_calls=None if BENCH is None else CALLS, cfg=CFG)
save_json(EXPORT / "posttrain_report.json", report); save_json(EXPORT / "self_correction_log.json", guardian.log)
sums = {f: sha256(EXPORT / f) for f in ("detector_state.pt", "detector_config.json")}
(EXPORT / "SHA256SUMS.txt").write_text("".join(f"{h}  {f}\n" for f, h in sums.items()))
print("Saved to", EXPORT); print(json.dumps(sums, indent=1))
print(json.dumps({k: meta[k] for k in ("logit_threshold_eer", "logit_threshold_1pct_fpr", "val_eer", "recommended_call_engine_env")}, indent=1))
print("RELEASE GATE:", "PASS" if RELEASE_OK else "NOT PASSED (keep V3)")

## 19. Next steps (on your Mac, in the repo)
1. **Only if the release gate passed:** download `export_posttrain_v4/` and copy `detector_state.pt` and `detector_config.json` into `backend/AI Deepfake Pipeline/`. Keep V3's files to roll back.
2. Re-pin: `python "backend/AI Deepfake Pipeline/model_integrity.py" --pin "backend/AI Deepfake Pipeline"` and review the diff.
3. Set the call-engine thresholds printed above (`recommended_call_engine_env`) as `THRESHOLD_ELEVATED_LOGIT` / `THRESHOLD_HIGH_LOGIT` on the agent.
4. Re-run the repo comparison with V4 added. Section 16 already reproduces it, but the repo run is the record. Check the **unseen voices only** rows.
5. Make at least one consented real call with background TV or music before demoing (`RECORD_CALLER_AUDIO=1` with everyone's permission).
6. Deploy as with V3: upload `detector_state.pt` as a new release asset and update the URL, file name and SHA-256 in `deploy/livekit/prepare.py`.

**What this run can't tell you:** the training corpora are read speech, and fakes come from MLAAD plus whatever commercial clips you supplied. If the external set isn't English, it may help less than the numbers on its own test split suggest. Compare those with the held-out benchmark voices. Real scam calls are conversational and go through real phone networks, so treat these as development results.